# Protein Backbone Diffusion V4a

V4a is a targeted continuation of V4 rather than a new architecture.
It keeps the working V4 denoiser and the stable V3b-style objective, then asks a narrower question:
can stronger global diagnostics explain the remaining over-collapsed samples, and can a lightweight sampling-time nonlocal C-alpha repulsion term reduce collapse without retraining or breaking local backbone geometry?


## 1. Runtime, setup, and V4a purpose

This notebook keeps the V4 data handling, normalization, chunking, masking, DDPM schedule, and geometry-regularized training objective unchanged.
The V4a additions are limited to:

- writing all new artifacts under `results/v4a/`
- loading an existing V4 or V4a checkpoint when available
- adding stronger global compactness diagnostics
- adding an optional sampling-time nonlocal CA anti-collapse guidance term
- comparing unguided and guided samples from the same checkpoint family


In [ ]:
import base64
import os
import random
import subprocess
import sys
import time
from getpass import getpass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch.utils.data import DataLoader

try:
    import google.colab  # type: ignore
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    drive = None  # type: ignore
    userdata = None  # type: ignore
    IN_COLAB = False

def env_flag(name: str, default: bool) -> bool:
    """Parse a boolean environment flag."""
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {'1', 'true', 'yes', 'y', 'on'}

def optional_int_env(name: str) -> int | None:
    """Parse an optional integer environment variable."""
    value = os.environ.get(name)
    if value is None or value.strip() == '':
        return None
    return int(value)

DRIVE_MOUNT_MODE = os.environ.get('BACKBONE_DIFFUSION_MOUNT_DRIVE', 'auto').strip().lower()
DRIVE_MOUNT_SKIPPED = DRIVE_MOUNT_MODE in {'0', 'false', 'no', 'off', 'skip'}
SAVE_TABLE_ARTIFACTS = env_flag('BACKBONE_DIFFUSION_SAVE_TABLES', not DRIVE_MOUNT_SKIPPED)

DRIVE_MOUNTPOINT = Path('/content/drive')
DRIVE_MYDRIVE = DRIVE_MOUNTPOINT / 'MyDrive'
DRIVE_MOUNTED = False

if IN_COLAB and drive is not None and not DRIVE_MOUNT_SKIPPED:
    try:
        drive.mount(str(DRIVE_MOUNTPOINT), force_remount=False)
        DRIVE_MOUNTED = DRIVE_MYDRIVE.exists()
    except Exception as exc:  # noqa: BLE001
        if DRIVE_MOUNT_MODE in {'1', 'true', 'yes', 'on', 'required'}:
            raise RuntimeError('Google Drive mounting was explicitly requested but failed.') from exc
        print(f'Google Drive mount skipped after failure: {exc}')
else:
    print('Google Drive mount skipped.')

if DRIVE_MOUNTED:
    print(f'Drive mounted: {DRIVE_MYDRIVE}')
else:
    print('Running without Google Drive mount; artifacts default to the repo-local results directory.')

REPO_REMOTE_URL = os.environ.get('GITHUB_REPO_URL', 'https://github.com/mitsenkov/latent-structure-diffusion.git')
REPO_BRANCH = os.environ.get('GITHUB_REPO_BRANCH', 'main')
REPO_CLONE_DIR = Path(os.environ.get('REPO_CLONE_DIR', '/content/latent-structure-diffusion'))
REPO_DRIVE_DIR = Path(os.environ.get('REPO_DRIVE_DIR', '/content/drive/MyDrive/latent-structure-diffusion'))
DEFAULT_CATH_FOLDER_ID = os.environ.get('CATH_SHARED_FOLDER_ID', '')
LOCAL_DATA_CACHE = Path(os.environ.get('CATH_LOCAL_CACHE', '/content/cath_backbone_data'))
ALLOW_GIT_CLONE = os.environ.get('ALLOW_GIT_CLONE', '1' if IN_COLAB else '0') == '1'

def get_github_token() -> str | None:
    """Return a GitHub token from env, Colab userdata, or an interactive prompt."""
    token = os.environ.get('GITHUB_TOKEN')
    if token:
        return token.strip()
    if IN_COLAB:
        try:
            token = userdata.get('GITHUB_TOKEN')
        except Exception:  # noqa: BLE001
            token = None
        if token:
            return str(token).strip()
    if ALLOW_GIT_CLONE:
        token = getpass('Paste the GitHub token with read access to this repo: ').strip()
        if token:
            return token
    return None

def find_repo_root(start_paths: list[Path] | None = None) -> Path | None:
    """Return the repository root if a checkout is already present."""
    candidates = start_paths or [Path.cwd().resolve(), REPO_CLONE_DIR, REPO_DRIVE_DIR]
    for start in candidates:
        if not start.exists():
            continue
        for candidate in [start, *start.parents]:
            if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
                return candidate
    return None

def build_auth_header(token: str) -> str:
    """Build a Git basic auth header for private GitHub clone access."""
    raw = f'x-access-token:{token}'.encode('utf-8')
    encoded = base64.b64encode(raw).decode('ascii')
    return f'AUTHORIZATION: basic {encoded}'

def bootstrap_repo() -> Path:
    """Make the repo available in Colab or reuse an existing checkout."""
    existing_root = find_repo_root()
    if existing_root is not None:
      if IN_COLAB:
          token = get_github_token()
          if token:
              pull_cmd = [
                  'git',
                  '-C',
                  str(existing_root),
                  '-c',
                  f'http.extraheader={build_auth_header(token)}',
                  'pull',
                  '--ff-only',
              ]
              subprocess.run(pull_cmd, check=True, capture_output=True, text=True)
      return existing_root

    if not IN_COLAB:
        raise FileNotFoundError(
            'Could not locate the repository root. Expected a folder containing pyproject.toml and src/.'
        )

    if not ALLOW_GIT_CLONE:
        raise FileNotFoundError(
            'Could not locate a local repo checkout. In Colab, set ALLOW_GIT_CLONE=1 or define GITHUB_TOKEN.'
        )

    token = get_github_token()
    if not token:
        raise FileNotFoundError(
            'GitHub cloning was enabled, but no token was provided.'
        )

    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    clone_cmd = [
        'git',
        '-c',
        f'http.extraheader={build_auth_header(token)}',
        'clone',
        '--branch',
        REPO_BRANCH,
        REPO_REMOTE_URL,
        str(REPO_CLONE_DIR),
    ]
    try:
        subprocess.run(clone_cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as exc:
        stderr = (exc.stderr or '').strip()
        stdout = (exc.stdout or '').strip()
        details = '\n'.join(part for part in [stdout, stderr] if part)
        raise RuntimeError(
            'GitHub clone failed in this Colab runtime. The token may be missing, invalid, or lack repo read access. '            f'Command: git clone --branch {REPO_BRANCH} {REPO_REMOTE_URL} {REPO_CLONE_DIR}\n{details}'
        ) from exc
    return REPO_CLONE_DIR

def ensure_gdown_installed() -> None:
    """Install gdown on demand."""
    try:
        import gdown  # type: ignore  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])

def resolve_data_dir() -> Path:
    """Resolve the dataset directory without requiring a Drive mount."""
    env_dir = os.environ.get('CATH_DATA_DIR')
    if env_dir:
        candidate = Path(env_dir)
        if (candidate / 'chain_set.jsonl').exists() and (candidate / 'chain_set_splits.json').exists():
            return candidate

    local_candidate = LOCAL_DATA_CACHE
    if (local_candidate / 'chain_set.jsonl').exists() and (local_candidate / 'chain_set_splits.json').exists():
        return local_candidate

    if IN_COLAB:
        ensure_gdown_installed()
        import gdown  # type: ignore

        local_candidate.mkdir(parents=True, exist_ok=True)
        url = f'https://drive.google.com/drive/folders/{DEFAULT_CATH_FOLDER_ID}'
        gdown.download_folder(url=url, output=str(local_candidate), quiet=False, use_cookies=False)
        if (local_candidate / 'chain_set.jsonl').exists() and (local_candidate / 'chain_set_splits.json').exists():
            return local_candidate

    raise FileNotFoundError(
        'Could not resolve the CATH dataset directory. Set CATH_DATA_DIR to a local path with chain_set.jsonl '
        'and chain_set_splits.json, or allow the notebook to download the public shared folder.'
    )

REPO_ROOT = bootstrap_repo()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = resolve_data_dir()

DEFAULT_ARTIFACT_DIR = (
    DRIVE_MYDRIVE / 'latent-structure-generation' / 'results'
    if DRIVE_MOUNTED
    else REPO_ROOT / 'results'
)
ARTIFACT_BASE_DIR = Path(os.environ.get('BACKBONE_DIFFUSION_ARTIFACT_BASE_DIR', str(DEFAULT_ARTIFACT_DIR)))
ARTIFACT_RUN_NAME = os.environ.get('BACKBONE_DIFFUSION_RUN_NAME', 'v4a')
ARTIFACT_DIR = ARTIFACT_BASE_DIR / ARTIFACT_RUN_NAME
FIGURE_DIR = ARTIFACT_DIR / 'figures'
TABLE_DIR = ARTIFACT_DIR / 'tables'
CHECKPOINT_DIR = ARTIFACT_DIR / 'checkpoints'
for directory in [FIGURE_DIR, TABLE_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42

REFERENCE_CHECKPOINT_FILENAME = os.environ.get('BACKBONE_DIFFUSION_REFERENCE_CHECKPOINT', 'best_backbone_diffusion.pt')
REFERENCE_CHECKPOINT_RUN_NAMES = [
    value.strip()
    for value in os.environ.get('BACKBONE_DIFFUSION_REFERENCE_RUN_NAMES', 'v4,v4a').split(',')
    if value.strip()
]

USE_NONLOCAL_CA_GUIDANCE = env_flag('BACKBONE_DIFFUSION_USE_NONLOCAL_CA_GUIDANCE', True)
GUIDANCE_SEQUENCE_SEPARATION = int(os.environ.get('BACKBONE_DIFFUSION_GUIDANCE_SEQUENCE_SEPARATION', '8'))
GUIDANCE_THRESHOLD_ANGSTROM = float(os.environ.get('BACKBONE_DIFFUSION_GUIDANCE_THRESHOLD_ANGSTROM', '8.0'))
GUIDANCE_SCALE = float(os.environ.get('BACKBONE_DIFFUSION_GUIDANCE_SCALE', '3e-4'))
GUIDANCE_START_TIMESTEP = optional_int_env('BACKBONE_DIFFUSION_GUIDANCE_START_TIMESTEP') or 75
GUIDANCE_END_TIMESTEP = optional_int_env('BACKBONE_DIFFUSION_GUIDANCE_END_TIMESTEP') or 10
V4A_SAMPLE_COUNT = int(os.environ.get('BACKBONE_DIFFUSION_V4A_SAMPLE_COUNT', '32'))
V4A_SAMPLE_BATCH_SIZE = int(os.environ.get('BACKBONE_DIFFUSION_V4A_SAMPLE_BATCH_SIZE', '8'))
V4A_SAMPLING_SEED = int(os.environ.get('BACKBONE_DIFFUSION_V4A_SAMPLING_SEED', str(SEED)))
V4A_NONLOCAL_SEQUENCE_SEPARATION = int(os.environ.get('BACKBONE_DIFFUSION_NONLOCAL_SEQUENCE_SEPARATION', '8'))
V4A_OPTIONAL_NONLOCAL_SEQUENCE_SEPARATION = int(os.environ.get('BACKBONE_DIFFUSION_OPTIONAL_NONLOCAL_SEQUENCE_SEPARATION', '16'))

def save_table_artifact(frame: pd.DataFrame, name: str, *, drop_columns: list[str] | None = None) -> dict[str, Path]:
    """Save a dataframe as CSV and Parquet under the run's tables directory."""
    if not SAVE_TABLE_ARTIFACTS:
        return {}
    export_frame = frame.drop(columns=[column for column in (drop_columns or []) if column in frame.columns]).copy()
    csv_path = TABLE_DIR / f'{name}.csv'
    parquet_path = TABLE_DIR / f'{name}.parquet'
    export_frame.to_csv(csv_path, index=False)
    try:
        export_frame.to_parquet(parquet_path, index=False)
    except Exception as exc:  # noqa: BLE001
        print(f'Could not write Parquet for {name}: {exc}')
    return {'csv': csv_path, 'parquet': parquet_path}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(0)
else:
    device_name = 'CPU'

try:
    import py3Dmol  # type: ignore
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'py3Dmol'])
    import py3Dmol  # type: ignore

if SAVE_TABLE_ARTIFACTS:
    try:
        import pyarrow  # type: ignore  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyarrow'])

from latent_structure_generation.backbone_diffusion import (
    BACKBONE_ATOMS,
    BackboneDataset,
    BackboneCoordinateEGNNDenoiser,
    BackboneNormalizationStats,
    apply_coordinate_normalisation,
    backbone_coords_to_protein,
    backbone_structure_summary,
    build_normalization_stats_from_dataframe,
    build_overlapping_backbone_chunks,
    collate_backbone_examples,
    centre_coordinates,
    create_noise_schedule,
    extract_backbone_from_coords_dict,
    flatten_backbone,
    invert_coordinate_normalisation,
    masked_coordinate_rmse,
    masked_noise_mse,
    pad_or_crop_backbone,
    predict_x0,
    q_sample,
    sample_backbone,
    sample_timesteps,
    structure_validity_report,
    to_pdb,
    unflatten_backbone,
)
from latent_structure_generation.plots import plot_ca_trace

print(f'Repository root: {REPO_ROOT}')
print(f'Data directory: {DATA_DIR}')
print(f'Artifacts: {ARTIFACT_DIR}')
print(f'Google Drive mounted: {DRIVE_MOUNTED}')
print(f'Table artifact saving enabled: {SAVE_TABLE_ARTIFACTS}')
print(f'Seed: {SEED}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Selected device: {device}')
print(f'Device name: {device_name}')
print(f'GitHub clone enabled: {ALLOW_GIT_CLONE}')
print(f'GitHub token present: {bool(os.environ.get("GITHUB_TOKEN") or (IN_COLAB and "GITHUB_TOKEN" in getattr(userdata, "keys", lambda: [])()))}')
print(f'Reference checkpoint run names: {REFERENCE_CHECKPOINT_RUN_NAMES}')
print(
    'Default sampling guidance config:',
    {
        'use_nonlocal_ca_guidance': USE_NONLOCAL_CA_GUIDANCE,
        'sequence_separation': GUIDANCE_SEQUENCE_SEPARATION,
        'threshold_angstrom': GUIDANCE_THRESHOLD_ANGSTROM,
        'scale': GUIDANCE_SCALE,
        'start_timestep': GUIDANCE_START_TIMESTEP,
        'end_timestep': GUIDANCE_END_TIMESTEP,
    },
)



## 2. Load the starter CATH tables

This reuses the provided Google Drive layout from the starter notebook. The split names are canonicalized to `train`, `validation`, and `test` so later code does not depend on `val` vs `validation` naming drift.


In [ ]:
chain_set_path = DATA_DIR / 'chain_set.jsonl'
split_path = DATA_DIR / 'chain_set_splits.json'

print(f'Reading {chain_set_path}')
df = pd.read_json(chain_set_path, lines=True)
print(f'Reading {split_path}')
chain_splits = pd.read_json(split_path, lines=True)

def canonical_split_name(name: str) -> str:
    """Map split labels to the canonical notebook names."""
    return {'val': 'validation', 'validation': 'validation', 'train': 'train', 'test': 'test'}.get(name, name)

split_lookup: dict[str, str] = {}

# Preserve the true train/validation/test buckets first.
for column in ['train', 'validation', 'test']:
    if column not in chain_splits.columns:
        continue
    canonical = canonical_split_name(column)
    values = chain_splits[column].iloc[0]
    for record_name in values:
        split_lookup.setdefault(record_name, canonical)

# cath_nodes is auxiliary metadata; use it only for records not already assigned.
if 'cath_nodes' in chain_splits.columns:
    cath_nodes = chain_splits['cath_nodes'].iloc[0]
    for record_name in cath_nodes.keys():
        split_lookup.setdefault(record_name, 'cath_nodes')

df['split'] = df['name'].map(split_lookup).fillna('unknown')
df['split'] = df['split'].map(canonical_split_name)

available_splits = sorted(df['split'].dropna().unique().tolist())
print('Available splits:', available_splits)
print('Split counts:')
print(df['split'].value_counts(dropna=False).sort_index())
print('Columns:', list(df.columns))
print('Example row keys:', list(df.iloc[0].index))

save_table_artifact(df, 'raw_chain_manifest', drop_columns=['coords'])


## 3. Data audit before training

This is the required review phase. It checks shapes, masks, invalid values, real lengths, truncation, coordinate ranges, backbone bond lengths, and a few visual examples before any training code runs.


In [ ]:
from collections import defaultdict

def move_batch_to_device(batch: dict[str, object], target_device: torch.device) -> dict[str, object]:
    """Move tensor values in a batch to the selected device."""
    moved: dict[str, object] = {}
    for key, value in batch.items():
        moved[key] = value.to(target_device) if torch.is_tensor(value) else value
    return moved

def audit_dataframe(split_df: pd.DataFrame, split_name: str, max_length: int) -> pd.DataFrame:
    """Audit one split and return a per-example summary table."""
    rows: list[dict[str, object]] = []
    for _, row in split_df.reset_index(drop=True).iterrows():
        record_id = row['name']
        try:
            coords, residue_mask = extract_backbone_from_coords_dict(row['coords'])
            raw_length = int(coords.shape[0])
            real_length = int(residue_mask.sum())
            coords_fixed, mask_fixed, truncated = pad_or_crop_backbone(coords, residue_mask, max_length)
            coords_t = torch.tensor(coords_fixed, dtype=torch.float32)
            mask_t = torch.tensor(mask_fixed.astype(np.float32), dtype=torch.float32)
            has_nan = bool(np.isnan(coords_fixed).any())
            has_inf = bool(np.isinf(coords_fixed).any())
            zero_real_residues = int(((np.abs(coords_fixed).sum(axis=(1, 2)) == 0.0) & mask_fixed).sum())
            summary = backbone_structure_summary(coords_t, mask_t)
            rows.append(
                {
                    'record_id': record_id,
                    'split': split_name,
                    'raw_length': raw_length,
                    'real_length': real_length,
                    'padded_length': int(mask_fixed.shape[0]),
                    'truncated': bool(truncated),
                    'has_nan': has_nan,
                    'has_inf': has_inf,
                    'zero_real_residues': zero_real_residues,
                    'keep_example': bool(real_length > 0 and not has_nan and not has_inf),
                    **summary,
                }
            )
        except Exception as exc:  # noqa: BLE001
            rows.append(
                {
                    'record_id': record_id,
                    'split': split_name,
                    'raw_length': np.nan,
                    'real_length': 0,
                    'padded_length': max_length,
                    'truncated': False,
                    'has_nan': True,
                    'has_inf': True,
                    'zero_real_residues': 0,
                    'keep_example': False,
                    'audit_error': str(exc),
                    'n_residues': 0,
                    'mean_adjacent_ca': np.nan,
                    'fraction_adjacent_ca_in_band': np.nan,
                    'mean_n_ca': np.nan,
                    'mean_ca_c': np.nan,
                    'mean_c_o': np.nan,
                    'mean_c_n': np.nan,
                    'radius_of_gyration': np.nan,
                }
            )
    return pd.DataFrame(rows)

def summarize_lengths(audit_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize real lengths, padding, and truncation for one split."""
    if audit_df.empty:
        return pd.DataFrame([
            {
                'split': 'unknown',
                'n_examples': 0,
                'min_real_length': np.nan,
                'median_real_length': np.nan,
                'max_real_length': np.nan,
                'mean_real_length': np.nan,
                'mean_padding_fraction': np.nan,
                'truncated_fraction': np.nan,
            }
        ])

    if 'keep_example' not in audit_df.columns:
        valid = audit_df.copy()
    else:
        valid = audit_df[audit_df['keep_example'].fillna(False)].copy()
    if valid.empty:
        split_name = audit_df['split'].iloc[0] if 'split' in audit_df.columns and len(audit_df) else 'unknown'
        return pd.DataFrame([
            {
                'split': split_name,
                'n_examples': 0,
                'min_real_length': np.nan,
                'median_real_length': np.nan,
                'max_real_length': np.nan,
                'mean_real_length': np.nan,
                'mean_padding_fraction': np.nan,
                'truncated_fraction': np.nan,
            }
        ])

    lengths = valid['real_length'].astype(int)
    padded_fraction = 1.0 - (lengths / valid['padded_length'].astype(int))
    return pd.DataFrame(
        [
            {
                'split': valid['split'].iloc[0],
                'n_examples': int(len(valid)),
                'min_real_length': int(lengths.min()),
                'median_real_length': float(lengths.median()),
                'max_real_length': int(lengths.max()),
                'mean_real_length': float(lengths.mean()),
                'mean_padding_fraction': float(padded_fraction.mean()),
                'truncated_fraction': float(valid['truncated'].mean()),
            }
        ]
    )

def coordinate_moments(split_df: pd.DataFrame, max_length: int) -> dict[str, np.ndarray | float]:
    """Compute mean/std/RMS statistics over real atom coordinates only."""
    sum_xyz = np.zeros(3, dtype=np.float64)
    sum_sq_xyz = np.zeros(3, dtype=np.float64)
    count = 0
    rg_values: list[float] = []
    for _, row in split_df.reset_index(drop=True).iterrows():
        coords, residue_mask = extract_backbone_from_coords_dict(row['coords'])
        if residue_mask.sum() == 0:
            continue
        coords_fixed, mask_fixed, _ = pad_or_crop_backbone(coords, residue_mask, max_length)
        coords_t = torch.tensor(coords_fixed, dtype=torch.float32)
        mask_t = torch.tensor(mask_fixed.astype(np.float32), dtype=torch.float32)
        centred = centre_coordinates(coords_t.unsqueeze(0), mask_t.unsqueeze(0))[0]
        real = centred[mask_t.bool()].reshape(-1, 3)
        sum_xyz += real.sum(dim=0).cpu().numpy()
        sum_sq_xyz += (real.pow(2)).sum(dim=0).cpu().numpy()
        count += int(real.shape[0])
        rg_values.append(float(backbone_structure_summary(coords_t, mask_t)['radius_of_gyration']))
    mean = sum_xyz / max(count, 1)
    var = sum_sq_xyz / max(count, 1) - mean**2
    std = np.sqrt(np.clip(var, 1e-6, None))
    return {'mean': mean, 'std': std, 'rms': float(np.sqrt(np.mean(sum_sq_xyz / max(count, 1)))), 'count': count, 'radius_of_gyration_mean': float(np.mean(rg_values)) if rg_values else float('nan')}

MAX_SEQ_LENGTH = 256
train_audit = audit_dataframe(df[df['split'] == 'train'], 'train', MAX_SEQ_LENGTH)
val_audit = audit_dataframe(df[df['split'] == 'validation'], 'validation', MAX_SEQ_LENGTH)
test_audit = audit_dataframe(df[df['split'] == 'test'], 'test', MAX_SEQ_LENGTH)

audit_df = pd.concat([train_audit, val_audit, test_audit], ignore_index=True)

length_summary_df = pd.concat(
    [summarize_lengths(train_audit), summarize_lengths(val_audit), summarize_lengths(test_audit)],
    ignore_index=True,
)

print('Audit summary by split:')
display(length_summary_df)

print('Invalid / filtered examples by split:')
filter_summary = (
    audit_df.groupby('split', as_index=False)
    .agg(total_examples=('record_id', 'count'), kept_examples=('keep_example', 'sum'), invalid_examples=('keep_example', lambda s: int((~s).sum())), truncated_examples=('truncated', 'sum'))
)
filter_summary['filtered_out'] = filter_summary['total_examples'] - filter_summary['kept_examples']
display(filter_summary)

print('Coordinate and structure summary on the training split:')
train_moments = coordinate_moments(df[df['split'] == 'train'], MAX_SEQ_LENGTH)
print(train_moments)

print('Head of the audit table:')
display(audit_df.head(8))

save_table_artifact(train_audit, 'train_audit')
save_table_artifact(val_audit, 'validation_audit')
save_table_artifact(test_audit, 'test_audit')
save_table_artifact(audit_df, 'audit_table')
save_table_artifact(length_summary_df, 'length_summary')
save_table_artifact(filter_summary, 'filter_summary')

print('Shape checks:')
print('coords expected shape: (B, L, 4, 3)')
print('mask expected shape: (B, L)')
print('backbone atom axis:', BACKBONE_ATOMS)
print('training split examples kept:', int(train_audit['keep_example'].sum()))
print('validation split examples kept:', int(val_audit['keep_example'].sum()))
print('test split examples kept:', int(test_audit['keep_example'].sum()))


### What the audit established

The audit keeps padded positions, drops only examples with no valid backbone residues, and reports the amount of truncation caused by fixed-length batching. The same summary tables also give the train-only statistics needed for normalization. Long training chains are then expanded into overlapping fixed-length chunks so we stop throwing away so much of each protein.


In [ ]:
# Build chain-level filtered split tables first.
train_chain_df = df[df['split'] == 'train'].reset_index(drop=True)
validation_chain_df = df[df['split'] == 'validation'].reset_index(drop=True)
test_chain_df = df[df['split'] == 'test'].reset_index(drop=True)

train_df = train_chain_df[train_audit['keep_example'].values].reset_index(drop=True)
validation_df = validation_chain_df[val_audit['keep_example'].values].reset_index(drop=True)
test_df = test_chain_df[test_audit['keep_example'].values].reset_index(drop=True)

# Expand long training chains into overlapping 256-residue windows.
TRAIN_CHUNK_STRIDE = 128
train_chunked_df = build_overlapping_backbone_chunks(
    train_df,
    split='train',
    max_length=MAX_SEQ_LENGTH,
    stride=TRAIN_CHUNK_STRIDE,
    record_id_column='name',
)

print('Filtered split sizes:')
print('train:', len(train_df))
print('validation:', len(validation_df))
print('test:', len(test_df))
print('train chunks:', len(train_chunked_df))
print('unique train parent chains:', train_chunked_df['parent_chain_id'].nunique())
print('training chunk stride:', TRAIN_CHUNK_STRIDE)
chunk_count_summary = train_chunked_df.groupby('parent_chain_id').size()
print('chunk count per parent chain: min', int(chunk_count_summary.min()), 'median', float(chunk_count_summary.median()), 'max', int(chunk_count_summary.max()))
print('chunk length range:', int(train_chunked_df['chunk_length'].min()), 'to', int(train_chunked_df['chunk_length'].max()))
chunk_count_summary_df = chunk_count_summary.rename_axis('parent_chain_id').reset_index(name='chunk_count')

save_table_artifact(train_df, 'train_chains_manifest', drop_columns=['coords'])
save_table_artifact(validation_df, 'validation_chains_manifest', drop_columns=['coords'])
save_table_artifact(test_df, 'test_chains_manifest', drop_columns=['coords'])
save_table_artifact(train_chunked_df, 'train_chunked_manifest', drop_columns=['coords'])
save_table_artifact(chunk_count_summary_df, 'train_chunk_counts')

OVERFIT_DEBUG = env_flag('BACKBONE_DIFFUSION_OVERFIT_DEBUG', False)
OVERFIT_SAMPLE_COUNT = int(os.environ.get('BACKBONE_DIFFUSION_OVERFIT_SAMPLE_COUNT', '128'))
OVERFIT_BATCH_SIZE = int(os.environ.get('BACKBONE_DIFFUSION_OVERFIT_BATCH_SIZE', '32'))
OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL = env_flag('BACKBONE_DIFFUSION_OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL', True)
OVERFIT_SHUFFLE_TRAIN = env_flag('BACKBONE_DIFFUSION_OVERFIT_SHUFFLE_TRAIN', False)

def build_overfit_subset(dataframe: pd.DataFrame, sample_count: int) -> pd.DataFrame:
    if sample_count <= 0:
        raise ValueError('BACKBONE_DIFFUSION_OVERFIT_SAMPLE_COUNT must be positive when overfit debug is enabled.')
    sample_n = min(sample_count, len(dataframe))
    return dataframe.sample(n=sample_n, random_state=SEED).reset_index(drop=True)

train_dataset_df = train_chunked_df
validation_dataset_df = validation_df
test_dataset_df = test_df
train_record_id_column = 'chunk_id'
validation_record_id_column = 'name'
test_record_id_column = 'name'
train_loader_shuffle = True

if OVERFIT_DEBUG:
    overfit_train_subset_df = build_overfit_subset(train_chunked_df, OVERFIT_SAMPLE_COUNT)
    train_dataset_df = overfit_train_subset_df
    train_loader_shuffle = OVERFIT_SHUFFLE_TRAIN
    if OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL:
        validation_dataset_df = overfit_train_subset_df.copy()
        validation_dataset_df['split'] = 'validation'
        test_dataset_df = overfit_train_subset_df.copy()
        test_dataset_df['split'] = 'test'
        validation_record_id_column = 'chunk_id'
        test_record_id_column = 'chunk_id'
    else:
        validation_dataset_df = validation_df.head(min(OVERFIT_SAMPLE_COUNT, len(validation_df))).reset_index(drop=True)
        test_dataset_df = test_df.head(min(OVERFIT_SAMPLE_COUNT, len(test_df))).reset_index(drop=True)

    print('OVERFIT DEBUG ENABLED')
    print('overfit train subset size:', len(train_dataset_df))
    print('overfit eval source:', 'train subset' if OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL else 'held-out validation/test head')
    save_table_artifact(train_dataset_df, 'v4a_overfit_train_subset_manifest', drop_columns=['coords'])
    if OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL:
        save_table_artifact(validation_dataset_df, 'v4a_overfit_eval_subset_manifest', drop_columns=['coords'])
else:
    print('Overfit debug disabled; using full train / validation / test datasets.')

train_dataset = BackboneDataset(train_dataset_df, split='train', max_length=MAX_SEQ_LENGTH, record_id_column=train_record_id_column)
validation_dataset = BackboneDataset(validation_dataset_df, split='validation', max_length=MAX_SEQ_LENGTH, record_id_column=validation_record_id_column)
test_dataset = BackboneDataset(test_dataset_df, split='test', max_length=MAX_SEQ_LENGTH, record_id_column=test_record_id_column)

DEFAULT_BATCH_SIZE = 32
BATCH_SIZE = min(OVERFIT_BATCH_SIZE, len(train_dataset)) if OVERFIT_DEBUG else DEFAULT_BATCH_SIZE
NUM_WORKERS = 2 if IN_COLAB else 0
PIN_MEMORY = device.type == 'cuda'

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=train_loader_shuffle,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=min(BATCH_SIZE, len(validation_dataset)),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=min(BATCH_SIZE, len(test_dataset)),
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_backbone_examples,
)

print('train dataset size used by loader:', len(train_dataset))
print('validation dataset size used by loader:', len(validation_dataset))
print('test dataset size used by loader:', len(test_dataset))
print('batch size:', BATCH_SIZE)
print('train loader shuffle:', train_loader_shuffle)

example_batch = next(iter(train_loader))
print('Example batch keys:', list(example_batch.keys()))
print('coords shape:', tuple(example_batch['coords'].shape), 'dtype:', example_batch['coords'].dtype, 'device:', example_batch['coords'].device)
print('mask shape:', tuple(example_batch['mask'].shape), 'dtype:', example_batch['mask'].dtype, 'device:', example_batch['mask'].device)
print('lengths shape:', tuple(example_batch['length'].shape))
print('real_length shape:', tuple(example_batch['real_length'].shape))
print('truncated shape:', tuple(example_batch['truncated'].shape))
print('first record ids:', example_batch['record_id'][:3])
print('first parent chain ids:', example_batch['parent_chain_id'][:3])
print('first chunk ids:', example_batch['chunk_id'][:3])


## 4. Visual sanity checks on real data

This shows a few training examples before any normalization or diffusion is applied. The point is to confirm that the backbone traces and masks line up with the expected protein geometry.


In [ ]:
def plot_ca_trace_from_batch(coords: torch.Tensor, mask: torch.Tensor, title: str):
    """Plot a CA trace using the existing plotting helper."""
    ca = coords[mask.bool(), 1, :].detach().cpu().numpy()
    fig, ax = plot_ca_trace(ca, title=title)
    plt.show()
    return fig, ax

def render_backbone_example(coords: torch.Tensor, mask: torch.Tensor, title: str):
    """Render one backbone structure as a PDB-backed py3Dmol viewer."""
    protein = backbone_coords_to_protein(coords, mask)
    pdb_str = to_pdb(protein)
    view = py3Dmol.view(width=700, height=500)
    view.addModel(pdb_str, 'pdb')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    view.show()
    return view

for index in range(min(2, example_batch['coords'].shape[0])):
    real_coords = example_batch['coords'][index]
    real_mask = example_batch['mask'][index]
    print(f'Real example {index}:', example_batch['record_id'][index])
    print(backbone_structure_summary(real_coords, real_mask))
    plot_ca_trace_from_batch(real_coords, real_mask.bool(), title=f"Real CA trace: {example_batch['record_id'][index]}")
    render_backbone_example(real_coords, real_mask, title=f"Real backbone: {example_batch['record_id'][index]}")


## 5. Preprocessing and train-only normalization

The model sees centered backbone coordinates that are standardized with statistics computed from the training split only. This cell also checks the flatten / unflatten / inverse-transform path before training starts.


In [ ]:
normalization_stats = build_normalization_stats_from_dataframe(train_df)
print('Normalization mean:', normalization_stats.mean)
print('Normalization std:', normalization_stats.std)

smoke_batch = next(iter(train_loader))
coords = smoke_batch['coords']
mask = smoke_batch['mask']
coords_centered = centre_coordinates(coords, mask)
coords_norm = apply_coordinate_normalisation(coords_centered, mask, normalization_stats)
coords_flat = flatten_backbone(coords_norm)
coords_roundtrip = unflatten_backbone(coords_flat)
coords_denorm = invert_coordinate_normalisation(coords_roundtrip, normalization_stats)

print('coords_centered:', tuple(coords_centered.shape))
print('coords_norm:', tuple(coords_norm.shape))
print('coords_flat:', tuple(coords_flat.shape))
print('coords_roundtrip:', tuple(coords_roundtrip.shape))
print('coords_denorm:', tuple(coords_denorm.shape))
print('max inverse diff:', float((coords_denorm - coords_centered).abs().max().item()))
print('smoke-test mask sum:', float(mask.sum().item()))


## 6. Diffusion utilities and model

V4 keeps the V3b geometry-augmented DDPM objective but changes the denoiser architecture. The new model treats each `N`, `CA`, `C`, and `O` atom as a graph node with atom-type identity, residue-position conditioning, timestep conditioning, and sparse backbone edges. EGNN-style message passing uses squared distances for hidden-state updates and relative-coordinate updates for equivariant coordinate refinement.


In [ ]:
TIMESTEPS = 100
noise_schedule = create_noise_schedule(TIMESTEPS, device=device)

MODEL_TYPE = 'BackboneCoordinateEGNNDenoiser'
MODEL_PREDICTION_HEAD = 'hidden_state_plus_coordinate_residual'
MODEL_HIDDEN_DIM = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_HIDDEN_DIM', '192'))
MODEL_NUM_LAYERS = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_NUM_LAYERS', '4'))
MODEL_TIME_EMBEDDING_DIM = int(os.environ.get('BACKBONE_DIFFUSION_MODEL_TIME_EMBEDDING_DIM', '128'))
MODEL_SEQUENCE_OFFSET_EDGES = tuple(int(value) for value in os.environ.get('BACKBONE_DIFFUSION_SEQUENCE_OFFSET_EDGES', '8,16').split(',') if value.strip())
LEARNING_RATE = float(os.environ.get('BACKBONE_DIFFUSION_LEARNING_RATE', '1e-3'))
model = BackboneCoordinateEGNNDenoiser(
    max_length=MAX_SEQ_LENGTH,
    hidden_dim=MODEL_HIDDEN_DIM,
    num_layers=MODEL_NUM_LAYERS,
    time_embedding_dim=MODEL_TIME_EMBEDDING_DIM,
    sequence_offset_edges=MODEL_SEQUENCE_OFFSET_EDGES,
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print('Model type:', MODEL_TYPE)
print('Model prediction head:', MODEL_PREDICTION_HEAD)
print('Model parameters:', sum(parameter.numel() for parameter in model.parameters()))
print('Learning rate:', LEARNING_RATE)
print('Sequence-offset CA edges:', MODEL_SEQUENCE_OFFSET_EDGES)
print('Schedule keys:', list(noise_schedule.keys()))
print('Schedule shapes:')
for key, value in noise_schedule.items():
    print(key, tuple(value.shape), value.device)

with torch.no_grad():
    demo_coords = coords_norm.to(device)
    demo_mask = mask.to(device)
    demo_t = sample_timesteps(demo_coords.shape[0], TIMESTEPS, device)
    demo_noise = torch.randn_like(flatten_backbone(demo_coords))
    demo_x0 = flatten_backbone(demo_coords)
    demo_xt = q_sample(demo_x0, demo_t, demo_noise, noise_schedule['alpha_bars'])
    demo_pred = model(demo_xt, demo_t, demo_mask)
    demo_noise_loss = masked_noise_mse(demo_pred, demo_noise, demo_mask)

print('x0 shape:', tuple(demo_x0.shape))
print('xt shape:', tuple(demo_xt.shape))
print('pred shape:', tuple(demo_pred.shape))
print('demo masked noise loss:', float(demo_noise_loss.detach().cpu()))
if getattr(model, 'last_forward_stats', None):
    print('Demo forward stats:', model.last_forward_stats)
assert demo_pred.shape == demo_x0.shape


## 7. Training, validation, and test evaluation

The loop tracks total objective, normal DDPM denoising loss, bond geometry loss, adjacent-CA geometry loss, and x0 reconstruction RMSE. Geometry losses are computed only after converting `x0_pred` back from normalized flattened coordinates into `B x L x 4 x 3` Angstrom coordinates.

V4 inherits the stable V3b objective directly: Smooth L1 geometry losses, `lambda_bond = 0.01`, `lambda_ca = 0.01`, geometry terms active only for timesteps `t <= 50`, and no radius anti-collapse term. The architecture change is the point of the run, not a new loss tweak.


In [ ]:
def prepare_batch_for_diffusion(batch: dict[str, object], target_device: torch.device, stats: BackboneNormalizationStats) -> tuple[torch.Tensor, torch.Tensor]:
    """Move a batch to device and return normalized flattened coords plus the mask."""
    batch = move_batch_to_device(batch, target_device)
    coords = batch['coords']
    mask = batch['mask']
    coords_centered = centre_coordinates(coords, mask)
    coords_norm = apply_coordinate_normalisation(coords_centered, mask, stats)
    coords_flat = flatten_backbone(coords_norm)
    return coords_flat, mask

NOISE_LOSS_WEIGHT = 1.0
LAMBDA_BOND = float(os.environ.get('BACKBONE_DIFFUSION_LAMBDA_BOND', '0.01'))
LAMBDA_CA = float(os.environ.get('BACKBONE_DIFFUSION_LAMBDA_CA', '0.01'))
GEOMETRY_LOSS_BETA = float(os.environ.get('BACKBONE_DIFFUSION_GEOMETRY_LOSS_BETA', '0.5'))
GEOMETRY_MAX_TIMESTEP = int(os.environ.get('BACKBONE_DIFFUSION_GEOMETRY_MAX_TIMESTEP', '50'))
GRAD_CLIP_NORM = float(os.environ.get('BACKBONE_DIFFUSION_GRAD_CLIP_NORM', '1.0'))
PRED_NEAR_ZERO_THRESHOLD = float(os.environ.get('BACKBONE_DIFFUSION_PRED_NEAR_ZERO_THRESHOLD', '1e-3'))
GEOMETRY_TARGETS_ANGSTROM = {
    'n_ca': 1.46,
    'ca_c': 1.53,
    'c_o': 1.23,
    'c_n': 1.33,
    'adjacent_ca': 3.80,
}

def geometry_timestep_mask(t: torch.Tensor, target_ndim: int) -> torch.Tensor:
    """Return a broadcastable mask that enables geometry loss only for selected timesteps."""
    active = (t <= GEOMETRY_MAX_TIMESTEP).float()
    view_shape = (t.shape[0],) + (1,) * (target_ndim - 1)
    return active.view(view_shape)

def masked_distance_huber(distances: torch.Tensor, target: float, valid_mask: torch.Tensor) -> torch.Tensor:
    """Smooth L1 distance error over valid bond or adjacent-residue positions."""
    valid = valid_mask.to(device=distances.device, dtype=distances.dtype)
    target_distances = torch.full_like(distances, fill_value=target)
    per_distance_loss = torch.nn.functional.smooth_l1_loss(
        distances,
        target_distances,
        beta=GEOMETRY_LOSS_BETA,
        reduction='none',
    )
    denom = valid.sum().clamp_min(1.0)
    return (per_distance_loss * valid).sum() / denom

def masked_tensor_rms(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Masked RMS over valid residues only."""
    mask_expanded = mask.unsqueeze(-1).expand_as(values).float()
    denom = mask_expanded.sum().clamp_min(1.0)
    return torch.sqrt(((values.pow(2) * mask_expanded).sum()) / denom)

def differentiable_predict_x0(
    x_t: torch.Tensor,
    t: torch.Tensor,
    pred_noise: torch.Tensor,
    alpha_bars: torch.Tensor,
) -> torch.Tensor:
    """Reconstruct x0 while preserving gradients for geometry losses."""
    alpha_bar_t = alpha_bars[t].view(-1, 1, 1).to(device=x_t.device, dtype=x_t.dtype)
    return (x_t - torch.sqrt(1.0 - alpha_bar_t) * pred_noise) / torch.sqrt(alpha_bar_t)

def x0_pred_to_angstrom_coords(x0_pred: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Convert normalized flattened x0 prediction to B x L x 4 x 3 Angstrom coordinates."""
    return invert_coordinate_normalisation(unflatten_backbone(x0_pred), stats)

def adjacent_ca_geometry_loss(x0_pred: torch.Tensor, mask: torch.Tensor, t: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Penalize adjacent CA distances at timesteps where x0 predictions are stable enough."""
    coords_angstrom = x0_pred_to_angstrom_coords(x0_pred, stats)
    ca = coords_angstrom[:, :, 1, :]
    adjacent_mask = mask[:, :-1].bool() & mask[:, 1:].bool()
    adjacent_mask = adjacent_mask.float() * geometry_timestep_mask(t, target_ndim=2).to(device=mask.device)
    adjacent_ca = torch.linalg.norm(ca[:, 1:, :] - ca[:, :-1, :], dim=-1)
    return masked_distance_huber(adjacent_ca, GEOMETRY_TARGETS_ANGSTROM['adjacent_ca'], adjacent_mask)

def backbone_bond_geometry_loss(x0_pred: torch.Tensor, mask: torch.Tensor, t: torch.Tensor, stats: BackboneNormalizationStats) -> torch.Tensor:
    """Penalize backbone bond lengths at timesteps where x0 predictions are stable enough."""
    coords_angstrom = x0_pred_to_angstrom_coords(x0_pred, stats)
    timestep_mask = geometry_timestep_mask(t, target_ndim=2).to(device=mask.device)
    residue_mask = mask.bool().float() * timestep_mask
    adjacent_mask = (mask[:, :-1].bool() & mask[:, 1:].bool()).float() * timestep_mask

    n_ca = torch.linalg.norm(coords_angstrom[:, :, 0, :] - coords_angstrom[:, :, 1, :], dim=-1)
    ca_c = torch.linalg.norm(coords_angstrom[:, :, 1, :] - coords_angstrom[:, :, 2, :], dim=-1)
    c_o = torch.linalg.norm(coords_angstrom[:, :, 2, :] - coords_angstrom[:, :, 3, :], dim=-1)
    c_n = torch.linalg.norm(coords_angstrom[:, :-1, 2, :] - coords_angstrom[:, 1:, 0, :], dim=-1)

    components = torch.stack([
        masked_distance_huber(n_ca, GEOMETRY_TARGETS_ANGSTROM['n_ca'], residue_mask),
        masked_distance_huber(ca_c, GEOMETRY_TARGETS_ANGSTROM['ca_c'], residue_mask),
        masked_distance_huber(c_o, GEOMETRY_TARGETS_ANGSTROM['c_o'], residue_mask),
        masked_distance_huber(c_n, GEOMETRY_TARGETS_ANGSTROM['c_n'], adjacent_mask),
    ])
    return components.mean()

def run_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer | None,
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
) -> dict[str, float]:
    """Run one training or evaluation epoch."""
    is_training = optimizer is not None
    model.train(is_training)
    totals = defaultdict(float)
    n_batches = 0

    for batch in loader:
        coords_flat, mask = prepare_batch_for_diffusion(batch, target_device, stats)
        t = sample_timesteps(coords_flat.shape[0], TIMESTEPS, target_device)
        noise = torch.randn_like(coords_flat)
        x_t = q_sample(coords_flat, t, noise, schedule['alpha_bars'])

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            pred_noise = model(x_t, t, mask)
            debug_stats = getattr(model, 'last_forward_stats', {})
            noise_loss = masked_noise_mse(pred_noise, noise, mask)
            x0_pred = differentiable_predict_x0(x_t, t, pred_noise, schedule['alpha_bars'])
            bond_loss = backbone_bond_geometry_loss(x0_pred, mask, t, stats)
            ca_loss = adjacent_ca_geometry_loss(x0_pred, mask, t, stats)
            total_loss = (NOISE_LOSS_WEIGHT * noise_loss) + (LAMBDA_BOND * bond_loss) + (LAMBDA_CA * ca_loss)
            recon_rmse = masked_coordinate_rmse(x0_pred, coords_flat, mask)
            grad_norm = 0.0
            if is_training:
                total_loss.backward()
                grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM))
                optimizer.step()

        totals['total_loss'] += float(total_loss.detach().cpu())
        totals['noise_loss'] += float(noise_loss.detach().cpu())
        totals['bond_geometry_loss'] += float(bond_loss.detach().cpu())
        totals['adjacent_ca_geometry_loss'] += float(ca_loss.detach().cpu())
        totals['x0_rmse'] += float(recon_rmse.detach().cpu())
        totals['pred_noise_rms'] += float(masked_tensor_rms(pred_noise, mask).detach().cpu())
        totals['x_t_rms'] += float(masked_tensor_rms(x_t, mask).detach().cpu())
        totals['x0_pred_rms'] += float(masked_tensor_rms(x0_pred, mask).detach().cpu())
        totals['near_zero_fraction'] += float(debug_stats.get(
            'near_zero_fraction',
            (((pred_noise.abs() < PRED_NEAR_ZERO_THRESHOLD).float() * mask.unsqueeze(-1).float()).sum() / mask.unsqueeze(-1).expand_as(pred_noise).float().sum().clamp_min(1.0)).detach().cpu(),
        ))
        totals['coord_residual_rms'] += float(debug_stats.get('coord_residual_rms', 0.0))
        totals['node_head_rms'] += float(debug_stats.get('node_head_rms', 0.0))
        totals['coord_update_rms'] += float(debug_stats.get('coord_update_rms', 0.0))
        totals['grad_norm'] += grad_norm
        n_batches += 1

    return {key: value / max(n_batches, 1) for key, value in totals.items()}

EPOCHS = int(os.environ.get('BACKBONE_DIFFUSION_EPOCHS', '5'))
history: list[dict[str, float]] = []
best_val_total_loss = float('inf')

print(
    f'Geometry loss settings: bond={LAMBDA_BOND}, adjacent_ca={LAMBDA_CA}, '
    f'beta={GEOMETRY_LOSS_BETA}, active_t<= {GEOMETRY_MAX_TIMESTEP}, radius=0.0'
)
print(
    f'Optimizer settings: lr={LEARNING_RATE}, grad_clip_norm={GRAD_CLIP_NORM}, '
    f'sequence_offset_edges={MODEL_SEQUENCE_OFFSET_EDGES}'
)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    train_metrics = run_epoch(model, train_loader, optimizer, normalization_stats, noise_schedule, device)
    val_metrics = run_epoch(model, validation_loader, None, normalization_stats, noise_schedule, device)
    test_metrics = run_epoch(model, test_loader, None, normalization_stats, noise_schedule, device)

    row = {
        'epoch': epoch,
        'train_total_loss': train_metrics['total_loss'],
        'train_noise_loss': train_metrics['noise_loss'],
        'train_bond_geometry_loss': train_metrics['bond_geometry_loss'],
        'train_adjacent_ca_geometry_loss': train_metrics['adjacent_ca_geometry_loss'],
        'train_x0_rmse': train_metrics['x0_rmse'],
        'train_pred_noise_rms': train_metrics['pred_noise_rms'],
        'train_x_t_rms': train_metrics['x_t_rms'],
        'train_x0_pred_rms': train_metrics['x0_pred_rms'],
        'train_near_zero_fraction': train_metrics['near_zero_fraction'],
        'train_coord_residual_rms': train_metrics['coord_residual_rms'],
        'train_node_head_rms': train_metrics['node_head_rms'],
        'train_coord_update_rms': train_metrics['coord_update_rms'],
        'train_grad_norm': train_metrics['grad_norm'],
        'validation_total_loss': val_metrics['total_loss'],
        'validation_noise_loss': val_metrics['noise_loss'],
        'validation_bond_geometry_loss': val_metrics['bond_geometry_loss'],
        'validation_adjacent_ca_geometry_loss': val_metrics['adjacent_ca_geometry_loss'],
        'validation_x0_rmse': val_metrics['x0_rmse'],
        'validation_pred_noise_rms': val_metrics['pred_noise_rms'],
        'validation_x_t_rms': val_metrics['x_t_rms'],
        'validation_x0_pred_rms': val_metrics['x0_pred_rms'],
        'validation_near_zero_fraction': val_metrics['near_zero_fraction'],
        'validation_coord_residual_rms': val_metrics['coord_residual_rms'],
        'validation_node_head_rms': val_metrics['node_head_rms'],
        'validation_coord_update_rms': val_metrics['coord_update_rms'],
        'test_total_loss': test_metrics['total_loss'],
        'test_noise_loss': test_metrics['noise_loss'],
        'test_bond_geometry_loss': test_metrics['bond_geometry_loss'],
        'test_adjacent_ca_geometry_loss': test_metrics['adjacent_ca_geometry_loss'],
        'test_x0_rmse': test_metrics['x0_rmse'],
        'test_pred_noise_rms': test_metrics['pred_noise_rms'],
        'test_x_t_rms': test_metrics['x_t_rms'],
        'test_x0_pred_rms': test_metrics['x0_pred_rms'],
        'test_near_zero_fraction': test_metrics['near_zero_fraction'],
        'test_coord_residual_rms': test_metrics['coord_residual_rms'],
        'test_node_head_rms': test_metrics['node_head_rms'],
        'test_coord_update_rms': test_metrics['coord_update_rms'],
        'lr': optimizer.param_groups[0]['lr'],
        'epoch_seconds': time.time() - start_time,
    }
    history.append(row)
    if row['validation_total_loss'] < best_val_total_loss:
        best_val_total_loss = row['validation_total_loss']
        torch.save(
            {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'normalization_mean': normalization_stats.mean,
                'normalization_std': normalization_stats.std,
                'timesteps': TIMESTEPS,
                'max_length': MAX_SEQ_LENGTH,
                'batch_size': BATCH_SIZE,
                'seed': SEED,
                'model_type': MODEL_TYPE,
                'model_prediction_head': MODEL_PREDICTION_HEAD,
                'learning_rate': LEARNING_RATE,
                'grad_clip_norm': GRAD_CLIP_NORM,
                'noise_loss_weight': NOISE_LOSS_WEIGHT,
                'lambda_bond': LAMBDA_BOND,
                'lambda_ca': LAMBDA_CA,
                'geometry_loss_beta': GEOMETRY_LOSS_BETA,
                'geometry_max_timestep': GEOMETRY_MAX_TIMESTEP,
                'geometry_loss_type': 'smooth_l1',
                'geometry_targets_angstrom': GEOMETRY_TARGETS_ANGSTROM,
                'pred_near_zero_threshold': PRED_NEAR_ZERO_THRESHOLD,
                'model_hidden_dim': MODEL_HIDDEN_DIM,
                'model_num_layers': MODEL_NUM_LAYERS,
                'model_time_embedding_dim': MODEL_TIME_EMBEDDING_DIM,
                'model_sequence_offset_edges': MODEL_SEQUENCE_OFFSET_EDGES,
                'saved_epoch': epoch,
                'saved_validation_total_loss': row['validation_total_loss'],
                'saved_validation_noise_loss': row['validation_noise_loss'],
            },
            CHECKPOINT_DIR / 'best_backbone_diffusion.pt',
        )
    print(
        f"Epoch {epoch:02d} | train total {row['train_total_loss']:.5f} | val total {row['validation_total_loss']:.5f} | "
        f"val noise {row['validation_noise_loss']:.5f} | val bond {row['validation_bond_geometry_loss']:.4f} | "
        f"val CA {row['validation_adjacent_ca_geometry_loss']:.4f} | train pred rms {row['train_pred_noise_rms']:.4f} | "
        f"val pred rms {row['validation_pred_noise_rms']:.4f} | val near-zero {row['validation_near_zero_fraction']:.3f} | "
        f"{row['epoch_seconds']:.1f}s"
    )

history_df = pd.DataFrame(history)
display(history_df)
save_table_artifact(history_df, 'backbone_diffusion_history')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(history_df['epoch'], history_df['train_total_loss'], label='Train total')
axes[0].plot(history_df['epoch'], history_df['validation_total_loss'], label='Validation total')
axes[0].plot(history_df['epoch'], history_df['test_total_loss'], label='Test total')
axes[0].plot(history_df['epoch'], history_df['validation_noise_loss'], linestyle='--', label='Validation noise')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('V4 training objective')
axes[0].legend()

axes[1].plot(history_df['epoch'], history_df['validation_bond_geometry_loss'], marker='o', label='Validation bond geometry')
axes[1].plot(history_df['epoch'], history_df['validation_adjacent_ca_geometry_loss'], marker='o', label='Validation adjacent CA')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Unweighted Smooth L1 geometry loss')
axes[1].set_title('V4 geometry losses')
axes[1].legend()

axes[2].plot(history_df['epoch'], history_df['train_pred_noise_rms'], marker='o', label='Train pred noise RMS')
axes[2].plot(history_df['epoch'], history_df['validation_pred_noise_rms'], marker='o', label='Validation pred noise RMS')
axes[2].plot(history_df['epoch'], history_df['validation_near_zero_fraction'], marker='s', linestyle='--', label='Validation near-zero frac')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('RMS / fraction')
axes[2].set_title('V4 output diagnostics')
axes[2].legend()

fig.tight_layout()
fig.savefig(FIGURE_DIR / 'backbone_diffusion_losses.png', dpi=150)
plt.show()


## 8. Sampling, checkpoint loading, and preview evaluation

V4a uses the same model definition and objective, but it prefers to reuse an existing compatible checkpoint from `results/v4a/` or `results/v4/` before any new sampling diagnostics are run.
If neither checkpoint exists, rerun section 7 to train the same V4-style model and write a fresh checkpoint under `results/v4a/checkpoints/`.


In [ ]:
def resolve_reference_checkpoint_path(model: torch.nn.Module, target_device: torch.device) -> tuple[Path, dict[str, object]]:
    """Return the first available checkpoint that is compatible with the current V4/V4a model definition."""
    searched_paths: list[Path] = []
    incompatible_paths: list[str] = []
    seen_paths: set[Path] = set()
    run_name_candidates = [*REFERENCE_CHECKPOINT_RUN_NAMES, ARTIFACT_RUN_NAME]
    for run_name in run_name_candidates:
        candidate = ARTIFACT_BASE_DIR / run_name / 'checkpoints' / REFERENCE_CHECKPOINT_FILENAME
        if candidate in seen_paths:
            continue
        seen_paths.add(candidate)
        searched_paths.append(candidate)
        if not candidate.exists():
            continue
        checkpoint = torch.load(candidate, map_location=target_device)
        try:
            model.load_state_dict(checkpoint['model_state_dict'])
        except RuntimeError as exc:
            incompatible_paths.append(f'{candidate} -> {exc}')
            continue
        return candidate, checkpoint
    searched = '\n'.join(str(path) for path in searched_paths)
    incompatible = '\n'.join(incompatible_paths)
    message = [
        'Could not find a compatible V4/V4a checkpoint.',
        f'Looked in:\n{searched}',
    ]
    if incompatible:
        message.append(f'Found incompatible checkpoints:\n{incompatible}')
    message.append('Train section 7, or place a compatible checkpoint under results/<run>/checkpoints/.')
    raise FileNotFoundError('\n'.join(message))

checkpoint_path, checkpoint = resolve_reference_checkpoint_path(model, device)
loaded_checkpoint_run_name = checkpoint_path.parent.parent.name
print(
    f"Loaded checkpoint from {checkpoint_path} | run={loaded_checkpoint_run_name} | "
    f"epoch={checkpoint.get('saved_epoch', 'unknown')} | "
    f"val_total={checkpoint.get('saved_validation_total_loss', 'unknown')} | "
    f"val_noise={checkpoint.get('saved_validation_noise_loss', 'unknown')}"
)

model.eval()
with torch.no_grad():
    eval_batch = next(iter(validation_loader))
    sample_mask = eval_batch['mask'][:4].to(device)
    sampled_flat = sample_backbone(
        model,
        noise_schedule,
        shape=(sample_mask.shape[0], MAX_SEQ_LENGTH, 12),
        mask=sample_mask,
        device=device,
    )
    sampled_coords_norm = unflatten_backbone(sampled_flat)
    sampled_coords = invert_coordinate_normalisation(sampled_coords_norm, normalization_stats)

    real_coords = eval_batch['coords'][:4]
    real_mask = eval_batch['mask'][:4]

real_rows = []
generated_rows = []
for index in range(sampled_coords.shape[0]):
    real_rows.append({'kind': 'real', 'index': index, **backbone_structure_summary(real_coords[index], real_mask[index])})
    generated_rows.append({'kind': 'generated', 'index': index, **backbone_structure_summary(sampled_coords[index].cpu(), sample_mask[index].cpu())})

real_eval_df = pd.DataFrame(real_rows)
generated_eval_df = pd.DataFrame(generated_rows)

display(real_eval_df)
display(generated_eval_df)

comparison_df = pd.concat([real_eval_df, generated_eval_df], ignore_index=True)
comparison_summary = comparison_df.groupby('kind', as_index=False).mean(numeric_only=True)
display(comparison_summary)
save_table_artifact(real_eval_df, 'v4a_preview_real_eval_metrics')
save_table_artifact(generated_eval_df, 'v4a_preview_generated_eval_metrics')
save_table_artifact(comparison_df, 'v4a_preview_real_vs_generated_summary')
save_table_artifact(comparison_summary, 'v4a_preview_real_vs_generated_summary_mean')


In [ ]:
for index in range(min(2, sampled_coords.shape[0])):
    print(f'Generated example {index}')
    print(backbone_structure_summary(sampled_coords[index].cpu(), sample_mask[index].cpu()))
    plot_ca_trace_from_batch(sampled_coords[index].cpu(), sample_mask[index].bool().cpu(), title=f'Generated CA trace {index}')
    render_backbone_example(sampled_coords[index].cpu(), sample_mask[index].cpu(), title=f'Generated backbone {index}')


## 9. V4a diagnostic evaluation

V4a keeps the trained V4 architecture and the V3b-style geometry objective unchanged.
The new question is narrower and more diagnostic: V4 now learns much better local geometry, but sampled structures are still too compact globally.
This section measures that collapse directly and then tests whether a lightweight sampling-time nonlocal CA repulsion term can reduce it without retraining.


In [ ]:
model_parameter_count = sum(parameter.numel() for parameter in model.parameters())
train_chunk_count_summary = train_chunked_df.groupby('parent_chain_id').size()
history_available = 'history_df' in globals() and isinstance(history_df, pd.DataFrame) and not history_df.empty
if history_available:
    best_epoch_row = history_df.loc[history_df['validation_total_loss'].idxmin()].to_dict()
else:
    best_epoch_row = {
        'epoch': checkpoint.get('saved_epoch', pd.NA),
        'validation_total_loss': checkpoint.get('saved_validation_total_loss', np.nan),
        'validation_noise_loss': checkpoint.get('saved_validation_noise_loss', np.nan),
        'test_total_loss': np.nan,
        'test_noise_loss': np.nan,
    }

def config_row(key: str, value: object) -> dict[str, object]:
    """Return a Parquet-safe config row with stable column types."""
    is_bool = isinstance(value, bool)
    is_int = isinstance(value, int) and not is_bool
    is_float = isinstance(value, float)
    return {
        'key': key,
        'value_text': str(value),
        'value_int': int(value) if is_int else pd.NA,
        'value_float': float(value) if (is_int or is_float) else np.nan,
    }

run_config_summary = pd.DataFrame([
    config_row('artifact_run_name', ARTIFACT_RUN_NAME),
    config_row('reference_checkpoint_run_name', loaded_checkpoint_run_name),
    config_row('reference_checkpoint_path', str(checkpoint_path)),
    config_row('seed', SEED),
    config_row('device', str(device)),
    config_row('device_name', device_name),
    config_row('max_seq_length', MAX_SEQ_LENGTH),
    config_row('backbone_atoms', ','.join(BACKBONE_ATOMS)),
    config_row('train_chains', len(train_df)),
    config_row('validation_chains', len(validation_df)),
    config_row('test_chains', len(test_df)),
    config_row('train_chunks', len(train_chunked_df)),
    config_row('train_chunk_stride', TRAIN_CHUNK_STRIDE),
    config_row('median_chunks_per_train_chain', float(train_chunk_count_summary.median())),
    config_row('max_chunks_per_train_chain', int(train_chunk_count_summary.max())),
    config_row('overfit_debug', OVERFIT_DEBUG),
    config_row('overfit_sample_count', OVERFIT_SAMPLE_COUNT),
    config_row('overfit_batch_size', OVERFIT_BATCH_SIZE),
    config_row('overfit_use_train_subset_for_eval', OVERFIT_USE_TRAIN_SUBSET_FOR_EVAL),
    config_row('overfit_shuffle_train', OVERFIT_SHUFFLE_TRAIN),
    config_row('train_dataset_size_used', len(train_dataset)),
    config_row('validation_dataset_size_used', len(validation_dataset)),
    config_row('test_dataset_size_used', len(test_dataset)),
    config_row('timesteps', TIMESTEPS),
    config_row('epochs', EPOCHS),
    config_row('batch_size', BATCH_SIZE),
    config_row('optimizer', 'Adam'),
    config_row('learning_rate', LEARNING_RATE),
    config_row('grad_clip_norm', GRAD_CLIP_NORM),
    config_row('model_type', MODEL_TYPE),
    config_row('model_prediction_head', MODEL_PREDICTION_HEAD),
    config_row('model_hidden_dim', MODEL_HIDDEN_DIM),
    config_row('model_num_layers', MODEL_NUM_LAYERS),
    config_row('model_time_embedding_dim', MODEL_TIME_EMBEDDING_DIM),
    config_row('model_sequence_offset_edges', ','.join(str(value) for value in MODEL_SEQUENCE_OFFSET_EDGES)),
    config_row('model_parameter_count', model_parameter_count),
    config_row('noise_loss_weight', NOISE_LOSS_WEIGHT),
    config_row('lambda_bond', LAMBDA_BOND),
    config_row('lambda_ca', LAMBDA_CA),
    config_row('lambda_radius_of_gyration', 0.0),
    config_row('geometry_loss_type', 'smooth_l1'),
    config_row('geometry_loss_beta', GEOMETRY_LOSS_BETA),
    config_row('geometry_max_timestep', GEOMETRY_MAX_TIMESTEP),
    config_row('pred_near_zero_threshold', PRED_NEAR_ZERO_THRESHOLD),
    config_row('target_n_ca_angstrom', GEOMETRY_TARGETS_ANGSTROM['n_ca']),
    config_row('target_ca_c_angstrom', GEOMETRY_TARGETS_ANGSTROM['ca_c']),
    config_row('target_c_o_angstrom', GEOMETRY_TARGETS_ANGSTROM['c_o']),
    config_row('target_c_n_angstrom', GEOMETRY_TARGETS_ANGSTROM['c_n']),
    config_row('target_adjacent_ca_angstrom', GEOMETRY_TARGETS_ANGSTROM['adjacent_ca']),
    config_row('use_nonlocal_ca_guidance_default', USE_NONLOCAL_CA_GUIDANCE),
    config_row('guidance_sequence_separation_default', GUIDANCE_SEQUENCE_SEPARATION),
    config_row('guidance_threshold_angstrom_default', GUIDANCE_THRESHOLD_ANGSTROM),
    config_row('guidance_scale_default', GUIDANCE_SCALE),
    config_row('guidance_start_timestep_default', GUIDANCE_START_TIMESTEP if GUIDANCE_START_TIMESTEP is not None else 'None'),
    config_row('guidance_end_timestep_default', GUIDANCE_END_TIMESTEP if GUIDANCE_END_TIMESTEP is not None else 'None'),
    config_row('v4a_sample_count', V4A_SAMPLE_COUNT),
    config_row('v4a_sample_batch_size', V4A_SAMPLE_BATCH_SIZE),
    config_row('v4a_sampling_seed', V4A_SAMPLING_SEED),
    config_row('v4a_nonlocal_sequence_separation', V4A_NONLOCAL_SEQUENCE_SEPARATION),
    config_row('v4a_optional_nonlocal_sequence_separation', V4A_OPTIONAL_NONLOCAL_SEQUENCE_SEPARATION),
    config_row('best_epoch', best_epoch_row['epoch']),
    config_row('best_validation_total_loss', float(best_epoch_row['validation_total_loss'])),
    config_row('best_validation_noise_loss', float(best_epoch_row['validation_noise_loss'])),
    config_row('best_test_total_loss_at_reported_epoch', float(best_epoch_row['test_total_loss'])),
    config_row('best_test_noise_loss_at_reported_epoch', float(best_epoch_row['test_noise_loss'])),
])
run_config_summary['value_int'] = run_config_summary['value_int'].astype('Int64')

display(run_config_summary)
save_table_artifact(run_config_summary, 'v4a_run_config_summary')


### Fixed-timestep denoising diagnostics

The training objective is unchanged from V4.
This diagnostic is kept so V4a can still confirm that any sampling-time guidance comparison is being run on a denoiser whose fixed-timestep behavior is understood, rather than changing the training objective under the hood.


In [ ]:
@torch.no_grad()
def evaluate_fixed_timesteps(
    model: torch.nn.Module,
    loader: DataLoader,
    split_name: str,
    timesteps_to_check: list[int],
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
    max_batches: int | None = None,
) -> pd.DataFrame:
    model.eval()
    rows = []
    for timestep in timesteps_to_check:
        totals = defaultdict(float)
        n_batches = 0
        for batch_index, batch in enumerate(loader):
            if max_batches is not None and batch_index >= max_batches:
                break
            coords_flat, mask = prepare_batch_for_diffusion(batch, target_device, stats)
            t = torch.full((coords_flat.shape[0],), timestep, device=target_device, dtype=torch.long)
            noise = torch.randn_like(coords_flat)
            x_t = q_sample(coords_flat, t, noise, schedule['alpha_bars'])
            pred_noise = model(x_t, t, mask)
            debug_stats = getattr(model, 'last_forward_stats', {})
            noise_loss = masked_noise_mse(pred_noise, noise, mask)
            x0_pred = differentiable_predict_x0(x_t, t, pred_noise, schedule['alpha_bars'])
            bond_loss = backbone_bond_geometry_loss(x0_pred, mask, t, stats)
            ca_loss = adjacent_ca_geometry_loss(x0_pred, mask, t, stats)
            total_loss = (NOISE_LOSS_WEIGHT * noise_loss) + (LAMBDA_BOND * bond_loss) + (LAMBDA_CA * ca_loss)
            recon_rmse = masked_coordinate_rmse(x0_pred, coords_flat, mask)
            totals['total_loss'] += float(total_loss.detach().cpu())
            totals['noise_loss'] += float(noise_loss.detach().cpu())
            totals['bond_geometry_loss'] += float(bond_loss.detach().cpu())
            totals['adjacent_ca_geometry_loss'] += float(ca_loss.detach().cpu())
            totals['x0_rmse'] += float(recon_rmse.detach().cpu())
            totals['pred_noise_rms'] += float(masked_tensor_rms(pred_noise, mask).detach().cpu())
            totals['x_t_rms'] += float(masked_tensor_rms(x_t, mask).detach().cpu())
            totals['x0_pred_rms'] += float(masked_tensor_rms(x0_pred, mask).detach().cpu())
            totals['near_zero_fraction'] += float(debug_stats.get('near_zero_fraction', 0.0))
            totals['coord_residual_rms'] += float(debug_stats.get('coord_residual_rms', 0.0))
            totals['node_head_rms'] += float(debug_stats.get('node_head_rms', 0.0))
            totals['coord_update_rms'] += float(debug_stats.get('coord_update_rms', 0.0))
            n_batches += 1
        rows.append({
            'split': split_name,
            'timestep': timestep,
            'noise_fraction': timestep / TIMESTEPS,
            'total_loss': totals['total_loss'] / max(n_batches, 1),
            'noise_loss': totals['noise_loss'] / max(n_batches, 1),
            'bond_geometry_loss': totals['bond_geometry_loss'] / max(n_batches, 1),
            'adjacent_ca_geometry_loss': totals['adjacent_ca_geometry_loss'] / max(n_batches, 1),
            'x0_rmse': totals['x0_rmse'] / max(n_batches, 1),
            'pred_noise_rms': totals['pred_noise_rms'] / max(n_batches, 1),
            'x_t_rms': totals['x_t_rms'] / max(n_batches, 1),
            'x0_pred_rms': totals['x0_pred_rms'] / max(n_batches, 1),
            'near_zero_fraction': totals['near_zero_fraction'] / max(n_batches, 1),
            'coord_residual_rms': totals['coord_residual_rms'] / max(n_batches, 1),
            'node_head_rms': totals['node_head_rms'] / max(n_batches, 1),
            'coord_update_rms': totals['coord_update_rms'] / max(n_batches, 1),
            'n_batches': n_batches,
        })
    return pd.DataFrame(rows)

TIMESTEP_DIAGNOSTIC_POINTS = [1, 5, 10, 25, 50, 75, 100]
validation_timestep_df = evaluate_fixed_timesteps(
    model,
    validation_loader,
    'validation',
    TIMESTEP_DIAGNOSTIC_POINTS,
    normalization_stats,
    noise_schedule,
    device,
)
test_timestep_df = evaluate_fixed_timesteps(
    model,
    test_loader,
    'test',
    TIMESTEP_DIAGNOSTIC_POINTS,
    normalization_stats,
    noise_schedule,
    device,
)
timestep_diagnostics_df = pd.concat([validation_timestep_df, test_timestep_df], ignore_index=True)

display(timestep_diagnostics_df)
save_table_artifact(timestep_diagnostics_df, 'v4a_timestep_diagnostics')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for split_name, split_df in timestep_diagnostics_df.groupby('split'):
    axes[0].plot(split_df['timestep'], split_df['noise_loss'], marker='o', label=f'{split_name} noise')
    axes[1].plot(split_df['timestep'], split_df['bond_geometry_loss'], marker='o', label=f'{split_name} bond')
    axes[1].plot(split_df['timestep'], split_df['adjacent_ca_geometry_loss'], marker='s', linestyle='--', label=f'{split_name} CA')
    axes[2].plot(split_df['timestep'], split_df['pred_noise_rms'], marker='o', label=f'{split_name} pred rms')
    axes[2].plot(split_df['timestep'], split_df['near_zero_fraction'], marker='s', linestyle='--', label=f'{split_name} near-zero')
axes[0].set_xlabel('Diffusion timestep')
axes[0].set_ylabel('Masked noise MSE')
axes[0].set_title('V4a fixed-timestep denoising')
axes[0].legend()
axes[1].set_xlabel('Diffusion timestep')
axes[1].set_ylabel('Smooth L1 geometry loss')
axes[1].set_title('V4a fixed-timestep geometry')
axes[1].legend()
axes[2].set_xlabel('Diffusion timestep')
axes[2].set_ylabel('RMS / fraction')
axes[2].set_title('V4a fixed-timestep output diagnostics')
axes[2].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'v4a_timestep_diagnostics.png', dpi=150)
plt.show()


### Global collapse diagnostics and guided sampling comparison

V4a keeps the local geometry checks from earlier notebooks, but adds stronger global diagnostics centered on nonlocal CA distances.
The main idea is that collapse is better diagnosed as too many nonlocal sequence positions becoming too close in 3D, rather than by radius of gyration alone.
The guided sampling experiment therefore applies a small repulsive energy only to nonlocal CA pairs that fall below a threshold during reverse sampling.


In [ ]:
V4A_COLLAPSE_RADIUS_FRACTION = float(os.environ.get('BACKBONE_DIFFUSION_V4A_COLLAPSE_RADIUS_FRACTION', '0.5'))
V4A_POOR_CA_BAND_THRESHOLD = float(os.environ.get('BACKBONE_DIFFUSION_V4A_POOR_CA_BAND_THRESHOLD', '0.8'))
V4A_NEAR_CONTACT_THRESHOLDS = (4.0, 5.0, 6.0, 8.0, 10.0)
V4A_REPORT_SEQUENCE_SEPARATIONS = tuple(dict.fromkeys([V4A_NONLOCAL_SEQUENCE_SEPARATION, V4A_OPTIONAL_NONLOCAL_SEQUENCE_SEPARATION]))

V4A_GUIDANCE_EXPERIMENTS = [
    {
        'condition': 'baseline_v4',
        'use_guidance': False,
        'sequence_separation': GUIDANCE_SEQUENCE_SEPARATION,
        'threshold_angstrom': 0.0,
        'guidance_scale': 0.0,
        'guidance_start_timestep': GUIDANCE_START_TIMESTEP,
        'guidance_end_timestep': GUIDANCE_END_TIMESTEP,
    },
    {
        'condition': 'guided_thr6_scale1e-4',
        'use_guidance': True,
        'sequence_separation': GUIDANCE_SEQUENCE_SEPARATION,
        'threshold_angstrom': 6.0,
        'guidance_scale': 0.0001,
        'guidance_start_timestep': GUIDANCE_START_TIMESTEP,
        'guidance_end_timestep': GUIDANCE_END_TIMESTEP,
    },
    {
        'condition': 'guided_thr8_scale3e-4',
        'use_guidance': True,
        'sequence_separation': GUIDANCE_SEQUENCE_SEPARATION,
        'threshold_angstrom': 8.0,
        'guidance_scale': 0.0003,
        'guidance_start_timestep': GUIDANCE_START_TIMESTEP,
        'guidance_end_timestep': GUIDANCE_END_TIMESTEP,
    },
    {
        'condition': 'guided_thr10_scale1e-4',
        'use_guidance': True,
        'sequence_separation': GUIDANCE_SEQUENCE_SEPARATION,
        'threshold_angstrom': 10.0,
        'guidance_scale': 0.0001,
        'guidance_start_timestep': GUIDANCE_START_TIMESTEP,
        'guidance_end_timestep': GUIDANCE_END_TIMESTEP,
    },
]
V4A_REPRESENTATIVE_GUIDED_CONDITION = next(
    (config['condition'] for config in V4A_GUIDANCE_EXPERIMENTS if config['condition'] == 'guided_thr8_scale3e-4'),
    V4A_GUIDANCE_EXPERIMENTS[-1]['condition'],
)


def set_sampling_seed(seed: int) -> None:
    """Reset torch RNGs so each sampling condition uses the same noise path."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def collect_conditioning_masks(loader: DataLoader, target_count: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Collect a fixed number of real validation examples and masks for matched-condition sampling."""
    coords_batches = []
    mask_batches = []
    seen = 0
    for batch in loader:
        coords = batch['coords']
        mask = batch['mask']
        remaining = target_count - seen
        coords_batches.append(coords[:remaining])
        mask_batches.append(mask[:remaining])
        seen += min(coords.shape[0], remaining)
        if seen >= target_count:
            break
    if not coords_batches or not mask_batches:
        raise ValueError('Could not collect any conditioning masks from the loader.')
    return torch.cat(coords_batches, dim=0), torch.cat(mask_batches, dim=0)


def is_guidance_timestep_active(timestep: int, start_timestep: int | None, end_timestep: int | None) -> bool:
    """Return True when guidance should be applied on this reverse-diffusion step."""
    if start_timestep is not None and timestep > start_timestep:
        return False
    if end_timestep is not None and timestep < end_timestep:
        return False
    return True


def series_distribution_summary(values: pd.Series, prefix: str) -> dict[str, float]:
    """Return a compact distribution summary for one numeric series."""
    numeric = pd.to_numeric(values, errors='coerce').dropna()
    if numeric.empty:
        return {
            f'{prefix}_mean': np.nan,
            f'{prefix}_median': np.nan,
            f'{prefix}_std': np.nan,
            f'{prefix}_p10': np.nan,
            f'{prefix}_p25': np.nan,
            f'{prefix}_p75': np.nan,
            f'{prefix}_p90': np.nan,
        }
    return {
        f'{prefix}_mean': float(numeric.mean()),
        f'{prefix}_median': float(numeric.median()),
        f'{prefix}_std': float(numeric.std(ddof=0)),
        f'{prefix}_p10': float(numeric.quantile(0.10)),
        f'{prefix}_p25': float(numeric.quantile(0.25)),
        f'{prefix}_p75': float(numeric.quantile(0.75)),
        f'{prefix}_p90': float(numeric.quantile(0.90)),
    }


def pooled_distance_distribution_summary(distances: np.ndarray, prefix: str) -> dict[str, float]:
    """Return pooled nonlocal distance distribution statistics."""
    array = np.asarray(distances, dtype=float)
    array = array[np.isfinite(array)]
    if array.size == 0:
        return {
            f'{prefix}_mean': np.nan,
            f'{prefix}_median': np.nan,
            f'{prefix}_std': np.nan,
            f'{prefix}_p10': np.nan,
            f'{prefix}_p25': np.nan,
            f'{prefix}_p75': np.nan,
            f'{prefix}_p90': np.nan,
        }
    return {
        f'{prefix}_mean': float(array.mean()),
        f'{prefix}_median': float(np.median(array)),
        f'{prefix}_std': float(array.std()),
        f'{prefix}_p10': float(np.quantile(array, 0.10)),
        f'{prefix}_p25': float(np.quantile(array, 0.25)),
        f'{prefix}_p75': float(np.quantile(array, 0.75)),
        f'{prefix}_p90': float(np.quantile(array, 0.90)),
    }


def extract_nonlocal_ca_distances(
    coords: torch.Tensor,
    mask: torch.Tensor,
    sequence_separation: int,
) -> torch.Tensor:
    """Return upper-triangular nonlocal CA pair distances for one structure."""
    if coords.ndim != 3 or coords.shape[-2:] != (4, 3):
        raise ValueError('coords must have shape (L, 4, 3).')
    if mask.ndim != 1:
        raise ValueError('mask must have shape (L,).')

    valid_indices = mask.bool().nonzero(as_tuple=False).squeeze(-1)
    if valid_indices.numel() < 2:
        return coords.new_empty((0,))

    ca = coords[:, 1, :][valid_indices]
    dists = torch.cdist(ca.unsqueeze(0), ca.unsqueeze(0)).squeeze(0)
    seq_sep = (valid_indices[:, None] - valid_indices[None, :]).abs()
    upper_triangle = torch.triu(torch.ones_like(seq_sep, dtype=torch.bool), diagonal=1)
    nonlocal_mask = (seq_sep > sequence_separation) & upper_triangle
    if not torch.any(nonlocal_mask):
        return coords.new_empty((0,))
    return dists[nonlocal_mask]


def nonlocal_ca_repulsion_energy(
    coords: torch.Tensor,
    mask: torch.Tensor,
    sequence_separation: int = 8,
    threshold_angstrom: float = 8.0,
) -> torch.Tensor:
    """Penalize nonlocal CA pairs that are closer than the chosen threshold."""
    if coords.ndim != 4 or coords.shape[-2:] != (4, 3):
        raise ValueError('coords must have shape (B, L, 4, 3).')
    if mask.ndim != 2:
        raise ValueError('mask must have shape (B, L).')

    energies: list[torch.Tensor] = []
    for batch_index in range(coords.shape[0]):
        valid_indices = mask[batch_index].bool().nonzero(as_tuple=False).squeeze(-1)
        if valid_indices.numel() < 2:
            continue
        ca = coords[batch_index, valid_indices, 1, :]
        dists = torch.cdist(ca.unsqueeze(0), ca.unsqueeze(0)).squeeze(0)
        seq_sep = (valid_indices[:, None] - valid_indices[None, :]).abs()
        upper_triangle = torch.triu(torch.ones_like(seq_sep, dtype=torch.bool), diagonal=1)
        nonlocal_mask = (seq_sep > sequence_separation) & upper_triangle
        if not torch.any(nonlocal_mask):
            continue
        too_close = torch.clamp(threshold_angstrom - dists[nonlocal_mask], min=0.0)
        energies.append(too_close.pow(2).mean())

    if not energies:
        return coords.new_tensor(0.0)
    return torch.stack(energies).mean()


def compute_structure_global_diagnostics(
    coords: torch.Tensor,
    mask: torch.Tensor,
    sequence_separation: int,
    near_contact_thresholds: tuple[float, ...],
) -> dict[str, float]:
    """Compute local geometry plus global compactness diagnostics for one structure."""
    summary = backbone_structure_summary(coords, mask)
    mask_bool = mask.bool()
    valid_indices = mask_bool.nonzero(as_tuple=False).squeeze(-1)
    ca_valid = coords[:, 1, :][mask_bool]

    if valid_indices.numel() == 0:
        row = {
            **summary,
            'end_to_end_ca_distance': np.nan,
            'max_pairwise_ca_distance': np.nan,
            'nonlocal_sequence_separation': float(sequence_separation),
            'nonlocal_pair_count': 0.0,
            'nonlocal_ca_distance_mean': np.nan,
            'nonlocal_ca_distance_median': np.nan,
            'nonlocal_ca_distance_std': np.nan,
            'nonlocal_ca_distance_p10': np.nan,
            'nonlocal_ca_distance_p25': np.nan,
            'nonlocal_ca_distance_p75': np.nan,
            'nonlocal_ca_distance_p90': np.nan,
            'bond_target_mae': np.nan,
        }
        for threshold in near_contact_thresholds:
            threshold_label = int(threshold)
            row[f'count_nonlocal_ca_pairs_below_{threshold_label}A'] = 0.0
            row[f'fraction_nonlocal_ca_pairs_below_{threshold_label}A'] = np.nan
        return row

    if valid_indices.numel() > 1:
        end_to_end = torch.linalg.norm(ca_valid[-1] - ca_valid[0]).item()
        max_pairwise = torch.cdist(ca_valid.unsqueeze(0), ca_valid.unsqueeze(0)).amax().item()
    else:
        end_to_end = float('nan')
        max_pairwise = float('nan')

    nonlocal_distances = extract_nonlocal_ca_distances(coords, mask, sequence_separation)
    nonlocal_distances_cpu = nonlocal_distances.detach().cpu().numpy().astype(float)
    finite_nonlocal = nonlocal_distances_cpu[np.isfinite(nonlocal_distances_cpu)]
    if finite_nonlocal.size == 0:
        nonlocal_summary = {
            'nonlocal_pair_count': 0.0,
            'nonlocal_ca_distance_mean': np.nan,
            'nonlocal_ca_distance_median': np.nan,
            'nonlocal_ca_distance_std': np.nan,
            'nonlocal_ca_distance_p10': np.nan,
            'nonlocal_ca_distance_p25': np.nan,
            'nonlocal_ca_distance_p75': np.nan,
            'nonlocal_ca_distance_p90': np.nan,
        }
    else:
        nonlocal_summary = {
            'nonlocal_pair_count': float(finite_nonlocal.size),
            'nonlocal_ca_distance_mean': float(finite_nonlocal.mean()),
            'nonlocal_ca_distance_median': float(np.median(finite_nonlocal)),
            'nonlocal_ca_distance_std': float(finite_nonlocal.std()),
            'nonlocal_ca_distance_p10': float(np.quantile(finite_nonlocal, 0.10)),
            'nonlocal_ca_distance_p25': float(np.quantile(finite_nonlocal, 0.25)),
            'nonlocal_ca_distance_p75': float(np.quantile(finite_nonlocal, 0.75)),
            'nonlocal_ca_distance_p90': float(np.quantile(finite_nonlocal, 0.90)),
        }

    bond_target_mae = np.nanmean([
        abs(summary['mean_n_ca'] - GEOMETRY_TARGETS_ANGSTROM['n_ca']) if np.isfinite(summary['mean_n_ca']) else np.nan,
        abs(summary['mean_ca_c'] - GEOMETRY_TARGETS_ANGSTROM['ca_c']) if np.isfinite(summary['mean_ca_c']) else np.nan,
        abs(summary['mean_c_o'] - GEOMETRY_TARGETS_ANGSTROM['c_o']) if np.isfinite(summary['mean_c_o']) else np.nan,
        abs(summary['mean_c_n'] - GEOMETRY_TARGETS_ANGSTROM['c_n']) if np.isfinite(summary['mean_c_n']) else np.nan,
    ])

    row = {
        **summary,
        'end_to_end_ca_distance': float(end_to_end),
        'max_pairwise_ca_distance': float(max_pairwise),
        'nonlocal_sequence_separation': float(sequence_separation),
        'bond_target_mae': float(bond_target_mae),
        **nonlocal_summary,
    }
    for threshold in near_contact_thresholds:
        threshold_label = int(threshold)
        if finite_nonlocal.size == 0:
            row[f'count_nonlocal_ca_pairs_below_{threshold_label}A'] = 0.0
            row[f'fraction_nonlocal_ca_pairs_below_{threshold_label}A'] = np.nan
        else:
            below = finite_nonlocal < threshold
            row[f'count_nonlocal_ca_pairs_below_{threshold_label}A'] = float(below.sum())
            row[f'fraction_nonlocal_ca_pairs_below_{threshold_label}A'] = float(below.mean())
    return row


def build_diagnostics_frame(
    coords_batch: torch.Tensor,
    mask_batch: torch.Tensor,
    condition: str,
    sequence_separation: int,
    near_contact_thresholds: tuple[float, ...],
    guidance_config: dict[str, object] | None = None,
) -> pd.DataFrame:
    """Compute per-structure diagnostics for one batch of structures."""
    rows = []
    guidance_config = guidance_config or {}
    for index in range(coords_batch.shape[0]):
        row = compute_structure_global_diagnostics(
            coords_batch[index],
            mask_batch[index],
            sequence_separation=sequence_separation,
            near_contact_thresholds=near_contact_thresholds,
        )
        row.update({
            'condition': condition,
            'index': index,
            'use_nonlocal_ca_guidance': bool(guidance_config.get('use_guidance', False)),
            'guidance_sequence_separation': guidance_config.get('sequence_separation', np.nan),
            'guidance_threshold_angstrom': guidance_config.get('threshold_angstrom', np.nan),
            'guidance_scale': guidance_config.get('guidance_scale', np.nan),
            'guidance_start_timestep': guidance_config.get('guidance_start_timestep', np.nan),
            'guidance_end_timestep': guidance_config.get('guidance_end_timestep', np.nan),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def collect_pooled_nonlocal_distances(
    coords_batch: torch.Tensor,
    mask_batch: torch.Tensor,
    sequence_separation: int,
) -> np.ndarray:
    """Collect pooled nonlocal CA distances across a batch of structures."""
    distances = []
    for index in range(coords_batch.shape[0]):
        pair_distances = extract_nonlocal_ca_distances(coords_batch[index], mask_batch[index], sequence_separation)
        if pair_distances.numel() == 0:
            continue
        distances.append(pair_distances.detach().cpu())
    if not distances:
        return np.asarray([], dtype=np.float32)
    return torch.cat(distances, dim=0).numpy().astype(np.float32)


def sample_backbone_with_optional_nonlocal_guidance(
    model: torch.nn.Module,
    schedule: dict[str, torch.Tensor],
    shape: tuple[int, int, int],
    mask: torch.Tensor,
    stats: BackboneNormalizationStats,
    target_device: torch.device,
    guidance_config: dict[str, object],
) -> torch.Tensor:
    """Sample flattened backbone coordinates with optional nonlocal CA guidance."""
    if len(shape) != 3 or shape[-1] != 12:
        raise ValueError('shape must be (B, L, 12).')

    batch_size, seq_len, _ = shape
    mask = mask.to(device=target_device, dtype=torch.float32)
    x = torch.randn(shape, device=target_device)

    betas = schedule['betas'].to(target_device)
    alphas = schedule['alphas'].to(target_device)
    alpha_bars = schedule['alpha_bars'].to(target_device)
    posterior_variance = schedule['posterior_variance'].to(target_device)
    timesteps = int(betas.shape[0] - 1)

    use_guidance = bool(guidance_config.get('use_guidance', False))
    sequence_separation = int(guidance_config.get('sequence_separation', GUIDANCE_SEQUENCE_SEPARATION))
    threshold_angstrom = float(guidance_config.get('threshold_angstrom', GUIDANCE_THRESHOLD_ANGSTROM))
    guidance_scale = float(guidance_config.get('guidance_scale', GUIDANCE_SCALE))
    start_timestep = guidance_config.get('guidance_start_timestep', GUIDANCE_START_TIMESTEP)
    end_timestep = guidance_config.get('guidance_end_timestep', GUIDANCE_END_TIMESTEP)

    model.eval()
    for timestep in range(timesteps, 0, -1):
        t = torch.full((batch_size,), timestep, device=target_device, dtype=torch.long)
        guidance_is_active = use_guidance and guidance_scale > 0.0 and is_guidance_timestep_active(
            timestep,
            start_timestep=start_timestep,
            end_timestep=end_timestep,
        )

        if guidance_is_active:
            x_in = x.detach().requires_grad_(True)
            pred_noise = model(x_in, t, mask)
            x0_pred = differentiable_predict_x0(x_in, t, pred_noise, alpha_bars)
            coords_angstrom = x0_pred_to_angstrom_coords(x0_pred, stats)
            guidance_energy = nonlocal_ca_repulsion_energy(
                coords_angstrom,
                mask,
                sequence_separation=sequence_separation,
                threshold_angstrom=threshold_angstrom,
            )
            guidance_grad = torch.autograd.grad(guidance_energy, x_in, retain_graph=False, create_graph=False)[0]
            x_current = x_in.detach()
            pred_noise_current = pred_noise.detach()
            guidance_grad = guidance_grad.detach() * mask.unsqueeze(-1)
        else:
            with torch.no_grad():
                pred_noise_current = model(x, t, mask)
            x_current = x
            guidance_grad = None

        alpha_t = alphas[timestep]
        beta_t = betas[timestep]
        alpha_bar_t = alpha_bars[timestep]
        mean = (x_current - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * pred_noise_current) / torch.sqrt(alpha_t)
        if guidance_grad is not None:
            mean = mean - guidance_scale * guidance_grad

        if timestep > 1:
            noise = torch.randn_like(x_current)
            variance = posterior_variance[timestep]
            x = mean + torch.sqrt(variance.clamp_min(1e-20)) * noise
        else:
            x = mean

        x = x * mask.unsqueeze(-1)

    return x


def sample_condition(
    condition_config: dict[str, object],
    masks: torch.Tensor,
    stats: BackboneNormalizationStats,
    schedule: dict[str, torch.Tensor],
    target_device: torch.device,
    sample_batch_size: int,
) -> tuple[pd.DataFrame, torch.Tensor, np.ndarray]:
    """Sample one condition and return per-structure diagnostics plus pooled distances."""
    generated_coord_batches = []
    metric_batches = []
    for start in range(0, masks.shape[0], sample_batch_size):
        end = min(start + sample_batch_size, masks.shape[0])
        batch_mask = masks[start:end].to(target_device)
        sampled_flat = sample_backbone_with_optional_nonlocal_guidance(
            model,
            schedule,
            shape=(batch_mask.shape[0], MAX_SEQ_LENGTH, 12),
            mask=batch_mask,
            stats=stats,
            target_device=target_device,
            guidance_config=condition_config,
        )
        sampled_coords_norm = unflatten_backbone(sampled_flat)
        sampled_coords = invert_coordinate_normalisation(sampled_coords_norm, stats).cpu()
        generated_coord_batches.append(sampled_coords)
        metric_batches.append(
            build_diagnostics_frame(
                sampled_coords,
                masks[start:end],
                condition=condition_config['condition'],
                sequence_separation=V4A_NONLOCAL_SEQUENCE_SEPARATION,
                near_contact_thresholds=V4A_NEAR_CONTACT_THRESHOLDS,
                guidance_config=condition_config,
            )
        )

    generated_coords = torch.cat(generated_coord_batches, dim=0)
    metrics_df = pd.concat(metric_batches, ignore_index=True)
    pooled_distances = collect_pooled_nonlocal_distances(
        generated_coords,
        masks,
        sequence_separation=V4A_NONLOCAL_SEQUENCE_SEPARATION,
    )
    return metrics_df, generated_coords, pooled_distances


def build_condition_summary(
    metrics_df: pd.DataFrame,
    pooled_nonlocal_distances: np.ndarray,
    condition_config: dict[str, object],
) -> dict[str, object]:
    """Aggregate diagnostics for one real or sampled condition."""
    summary = {
        'condition': condition_config['condition'],
        'sample_count': int(len(metrics_df)),
        'use_nonlocal_ca_guidance': bool(condition_config.get('use_guidance', False)),
        'guidance_sequence_separation': condition_config.get('sequence_separation', np.nan),
        'guidance_threshold_angstrom': condition_config.get('threshold_angstrom', np.nan),
        'guidance_scale': condition_config.get('guidance_scale', np.nan),
        'guidance_start_timestep': condition_config.get('guidance_start_timestep', np.nan),
        'guidance_end_timestep': condition_config.get('guidance_end_timestep', np.nan),
        'collapse_count': int(metrics_df['collapse_flag'].sum()),
        'collapse_fraction': float(metrics_df['collapse_flag'].mean()),
        'poor_adjacent_ca_band_count': int(metrics_df['poor_adjacent_ca_band_flag'].sum()),
        'poor_adjacent_ca_band_fraction': float(metrics_df['poor_adjacent_ca_band_flag'].mean()),
        'mean_adjacent_ca': float(metrics_df['mean_adjacent_ca'].mean()),
        'median_adjacent_ca': float(metrics_df['mean_adjacent_ca'].median()),
        'fraction_adjacent_ca_in_band_mean': float(metrics_df['fraction_adjacent_ca_in_band'].mean()),
        'fraction_adjacent_ca_in_band_median': float(metrics_df['fraction_adjacent_ca_in_band'].median()),
        'mean_n_ca': float(metrics_df['mean_n_ca'].mean()),
        'mean_ca_c': float(metrics_df['mean_ca_c'].mean()),
        'mean_c_o': float(metrics_df['mean_c_o'].mean()),
        'mean_c_n': float(metrics_df['mean_c_n'].mean()),
        'bond_target_mae_mean': float(metrics_df['bond_target_mae'].mean()),
        'bond_target_mae_median': float(metrics_df['bond_target_mae'].median()),
        'max_pairwise_ca_distance_mean': float(metrics_df['max_pairwise_ca_distance'].mean()),
        'max_pairwise_ca_distance_median': float(metrics_df['max_pairwise_ca_distance'].median()),
        'end_to_end_ca_distance_mean': float(metrics_df['end_to_end_ca_distance'].mean()),
        'end_to_end_ca_distance_median': float(metrics_df['end_to_end_ca_distance'].median()),
        'median_nonlocal_ca_distance_mean': float(metrics_df['nonlocal_ca_distance_median'].mean()),
        'median_nonlocal_ca_distance_median': float(metrics_df['nonlocal_ca_distance_median'].median()),
        'p10_nonlocal_ca_distance_mean': float(metrics_df['nonlocal_ca_distance_p10'].mean()),
        'p10_nonlocal_ca_distance_median': float(metrics_df['nonlocal_ca_distance_p10'].median()),
        'fraction_nonlocal_ca_pairs_below_6A_mean': float(metrics_df['fraction_nonlocal_ca_pairs_below_6A'].mean()),
        'fraction_nonlocal_ca_pairs_below_8A_mean': float(metrics_df['fraction_nonlocal_ca_pairs_below_8A'].mean()),
        'fraction_nonlocal_ca_pairs_below_10A_mean': float(metrics_df['fraction_nonlocal_ca_pairs_below_10A'].mean()),
        'fraction_nonlocal_ca_pairs_below_4A_mean': float(metrics_df['fraction_nonlocal_ca_pairs_below_4A'].mean()),
        'fraction_nonlocal_ca_pairs_below_5A_mean': float(metrics_df['fraction_nonlocal_ca_pairs_below_5A'].mean()),
    }
    summary.update(series_distribution_summary(metrics_df['radius_of_gyration'], 'radius_of_gyration'))
    summary.update(pooled_distance_distribution_summary(pooled_nonlocal_distances, 'pooled_nonlocal_ca_distance'))
    return summary


def build_sequence_separation_summary(
    condition: str,
    coords_batch: torch.Tensor,
    mask_batch: torch.Tensor,
    sequence_separation: int,
) -> dict[str, object]:
    """Summarize pooled nonlocal CA distances for one sequence-separation setting."""
    pooled = collect_pooled_nonlocal_distances(coords_batch, mask_batch, sequence_separation)
    row = {
        'condition': condition,
        'sequence_separation': int(sequence_separation),
        'pair_count': int(len(pooled)),
        'fraction_nonlocal_ca_pairs_below_6A': float(np.mean(pooled < 6.0)) if len(pooled) else np.nan,
        'fraction_nonlocal_ca_pairs_below_8A': float(np.mean(pooled < 8.0)) if len(pooled) else np.nan,
        'fraction_nonlocal_ca_pairs_below_10A': float(np.mean(pooled < 10.0)) if len(pooled) else np.nan,
    }
    row.update(pooled_distance_distribution_summary(pooled, 'nonlocal_ca_distance'))
    return row


v4a_reference_sample_count = min(V4A_SAMPLE_COUNT, len(validation_dataset))
v4a_real_coords, v4a_sample_masks = collect_conditioning_masks(validation_loader, v4a_reference_sample_count)

v4a_real_condition_config = {
    'condition': 'real_validation_reference',
    'use_guidance': False,
    'sequence_separation': np.nan,
    'threshold_angstrom': np.nan,
    'guidance_scale': np.nan,
    'guidance_start_timestep': np.nan,
    'guidance_end_timestep': np.nan,
}
v4a_real_global_diagnostics_df = build_diagnostics_frame(
    v4a_real_coords,
    v4a_sample_masks,
    condition=v4a_real_condition_config['condition'],
    sequence_separation=V4A_NONLOCAL_SEQUENCE_SEPARATION,
    near_contact_thresholds=V4A_NEAR_CONTACT_THRESHOLDS,
    guidance_config=v4a_real_condition_config,
)
v4a_real_pooled_nonlocal_distances = collect_pooled_nonlocal_distances(
    v4a_real_coords,
    v4a_sample_masks,
    sequence_separation=V4A_NONLOCAL_SEQUENCE_SEPARATION,
)
real_rg_median = float(v4a_real_global_diagnostics_df['radius_of_gyration'].median())
collapse_radius_threshold = V4A_COLLAPSE_RADIUS_FRACTION * real_rg_median

v4a_real_global_diagnostics_df['collapse_flag'] = v4a_real_global_diagnostics_df['radius_of_gyration'] < collapse_radius_threshold
v4a_real_global_diagnostics_df['poor_adjacent_ca_band_flag'] = (
    v4a_real_global_diagnostics_df['fraction_adjacent_ca_in_band'] < V4A_POOR_CA_BAND_THRESHOLD
)

v4a_sample_runs: dict[str, dict[str, object]] = {}
for experiment in V4A_GUIDANCE_EXPERIMENTS:
    set_sampling_seed(V4A_SAMPLING_SEED)
    condition_metrics_df, condition_coords, pooled_distances = sample_condition(
        experiment,
        v4a_sample_masks,
        normalization_stats,
        noise_schedule,
        device,
        sample_batch_size=V4A_SAMPLE_BATCH_SIZE,
    )
    condition_metrics_df['collapse_flag'] = condition_metrics_df['radius_of_gyration'] < collapse_radius_threshold
    condition_metrics_df['poor_adjacent_ca_band_flag'] = (
        condition_metrics_df['fraction_adjacent_ca_in_band'] < V4A_POOR_CA_BAND_THRESHOLD
    )
    v4a_sample_runs[experiment['condition']] = {
        'config': experiment,
        'metrics_df': condition_metrics_df,
        'coords': condition_coords,
        'pooled_nonlocal_distances': pooled_distances,
    }
    print(
        f"Completed {experiment['condition']} | guidance={experiment['use_guidance']} | "
        f"threshold={experiment['threshold_angstrom']} | scale={experiment['guidance_scale']}"
    )

v4a_baseline_global_diagnostics_df = v4a_sample_runs['baseline_v4']['metrics_df'].copy()
v4a_guided_global_diagnostics_df = pd.concat(
    [
        run['metrics_df']
        for name, run in v4a_sample_runs.items()
        if name != 'baseline_v4'
    ],
    ignore_index=True,
)
v4a_all_sample_global_diagnostics_df = pd.concat(
    [run['metrics_df'] for run in v4a_sample_runs.values()],
    ignore_index=True,
)

v4a_guidance_summary_rows = [
    build_condition_summary(
        v4a_real_global_diagnostics_df,
        v4a_real_pooled_nonlocal_distances,
        v4a_real_condition_config,
    )
]
for condition_name, run in v4a_sample_runs.items():
    v4a_guidance_summary_rows.append(
        build_condition_summary(
            run['metrics_df'],
            run['pooled_nonlocal_distances'],
            run['config'],
        )
    )
v4a_guidance_summary_df = pd.DataFrame(v4a_guidance_summary_rows)

v4a_sequence_separation_summary_rows = [
    build_sequence_separation_summary(
        v4a_real_condition_config['condition'],
        v4a_real_coords,
        v4a_sample_masks,
        sequence_separation,
    )
    for sequence_separation in V4A_REPORT_SEQUENCE_SEPARATIONS
]
for condition_name, run in v4a_sample_runs.items():
    for sequence_separation in V4A_REPORT_SEQUENCE_SEPARATIONS:
        v4a_sequence_separation_summary_rows.append(
            build_sequence_separation_summary(
                condition_name,
                run['coords'],
                v4a_sample_masks,
                sequence_separation,
            )
        )
v4a_sequence_separation_summary_df = pd.DataFrame(v4a_sequence_separation_summary_rows)

for condition_name, run in v4a_sample_runs.items():
    condition_table_name = f"v4a_condition_{condition_name}_eval_metrics".replace('-', '_')
    save_table_artifact(run['metrics_df'], condition_table_name)

save_table_artifact(v4a_real_global_diagnostics_df, 'v4a_real_eval_metrics')
save_table_artifact(v4a_baseline_global_diagnostics_df, 'v4a_generated_eval_metrics')
save_table_artifact(v4a_real_global_diagnostics_df, 'v4a_global_diagnostics_real')
save_table_artifact(v4a_baseline_global_diagnostics_df, 'v4a_global_diagnostics_baseline_samples')
save_table_artifact(v4a_guided_global_diagnostics_df, 'v4a_global_diagnostics_guided_samples')
save_table_artifact(v4a_all_sample_global_diagnostics_df, 'v4a_global_diagnostics_all_samples')
save_table_artifact(v4a_guidance_summary_df, 'v4a_guidance_comparison_summary')
save_table_artifact(v4a_guidance_summary_df, 'v4a_real_vs_generated_metric_summary')
save_table_artifact(v4a_sequence_separation_summary_df, 'v4a_nonlocal_sequence_separation_summary')
save_table_artifact(pd.DataFrame(V4A_GUIDANCE_EXPERIMENTS), 'v4a_guidance_condition_config')

display(v4a_guidance_summary_df)
display(v4a_sequence_separation_summary_df)

baseline_row = v4a_guidance_summary_df[v4a_guidance_summary_df['condition'] == 'baseline_v4'].iloc[0]
guided_candidate_rows = v4a_guidance_summary_df[
    v4a_guidance_summary_df['condition'].isin([config['condition'] for config in V4A_GUIDANCE_EXPERIMENTS if config['use_guidance']])
].copy()
guided_candidate_rows['radius_gap_abs'] = (
    guided_candidate_rows['radius_of_gyration_median'] - v4a_guidance_summary_df.iloc[0]['radius_of_gyration_median']
).abs()
recommended_guided_row = guided_candidate_rows.sort_values(
    by=[
        'poor_adjacent_ca_band_count',
        'collapse_count',
        'fraction_nonlocal_ca_pairs_below_8A_mean',
        'bond_target_mae_mean',
        'radius_gap_abs',
    ],
    ascending=[True, True, True, True, True],
).iloc[0]

local_geometry_preserved = (
    recommended_guided_row['fraction_adjacent_ca_in_band_mean'] >= baseline_row['fraction_adjacent_ca_in_band_mean'] - 0.05
    and recommended_guided_row['bond_target_mae_mean'] <= baseline_row['bond_target_mae_mean'] + 0.05
)
if (
    recommended_guided_row['collapse_count'] < baseline_row['collapse_count']
    and recommended_guided_row['fraction_nonlocal_ca_pairs_below_8A_mean'] < baseline_row['fraction_nonlocal_ca_pairs_below_8A_mean']
    and local_geometry_preserved
):
    v4a_interpretation = 'Sampling-time nonlocal CA guidance reduced collapse and close contacts without a major local-geometry regression.'
elif (
    recommended_guided_row['collapse_count'] < baseline_row['collapse_count']
    and recommended_guided_row['fraction_nonlocal_ca_pairs_below_8A_mean'] < baseline_row['fraction_nonlocal_ca_pairs_below_8A_mean']
):
    v4a_interpretation = 'Guidance reduced collapse and close contacts, but the local-geometry trade-off should be checked carefully.'
elif local_geometry_preserved:
    v4a_interpretation = 'Guidance preserved local geometry, but the global anti-collapse effect was limited in this run.'
else:
    v4a_interpretation = 'Guidance did not clearly improve the global collapse/local geometry trade-off in this run.'

summary_lines = [
    '# V4a Summary',
    '',
    'V4a keeps the V4 denoiser and V3b-style geometry objective fixed, then evaluates global over-collapse directly with nonlocal CA diagnostics.',
    '',
    f'- Reference checkpoint run: `{loaded_checkpoint_run_name}` at `{checkpoint_path}`',
    f'- Sample count per condition: `{v4a_reference_sample_count}`',
    f'- Collapse threshold: `{collapse_radius_threshold:.2f} A` radius of gyration (`{V4A_COLLAPSE_RADIUS_FRACTION:.2f} x` real median `{real_rg_median:.2f} A`)',
    f'- Main nonlocal diagnostic separation: `{V4A_NONLOCAL_SEQUENCE_SEPARATION}` residues',
    '',
    '## Baseline V4',
    '',
    f"- Collapse count: `{int(baseline_row['collapse_count'])}` / `{int(baseline_row['sample_count'])}`",
    f"- Radius of gyration mean / median: `{baseline_row['radius_of_gyration_mean']:.2f} / {baseline_row['radius_of_gyration_median']:.2f} A`",
    f"- Pooled nonlocal CA median distance: `{baseline_row['pooled_nonlocal_ca_distance_median']:.2f} A`",
    f"- Mean fraction nonlocal CA pairs below 8 A: `{baseline_row['fraction_nonlocal_ca_pairs_below_8A_mean']:.4f}`",
    f"- Adjacent CA in-band mean: `{baseline_row['fraction_adjacent_ca_in_band_mean']:.4f}`",
    '',
    f"## Recommended guided condition: `{recommended_guided_row['condition']}`",
    '',
    f"- Collapse count: `{int(recommended_guided_row['collapse_count'])}` / `{int(recommended_guided_row['sample_count'])}`",
    f"- Radius of gyration mean / median: `{recommended_guided_row['radius_of_gyration_mean']:.2f} / {recommended_guided_row['radius_of_gyration_median']:.2f} A`",
    f"- Pooled nonlocal CA median distance: `{recommended_guided_row['pooled_nonlocal_ca_distance_median']:.2f} A`",
    f"- Mean fraction nonlocal CA pairs below 8 A: `{recommended_guided_row['fraction_nonlocal_ca_pairs_below_8A_mean']:.4f}`",
    f"- Adjacent CA in-band mean: `{recommended_guided_row['fraction_adjacent_ca_in_band_mean']:.4f}`",
    f"- Bond target MAE mean: `{recommended_guided_row['bond_target_mae_mean']:.4f} A`",
    '',
    '## Interpretation',
    '',
    f'- {v4a_interpretation}',
    '- Radius of gyration alone is not treated as the win condition; the main judgment also uses nonlocal CA distance distributions and close-contact fractions.',
    '- If guidance expands the structures by creating obvious outliers or worse local geometry, the nonlocal and bond metrics should overrule any apparent radius gain.',
    '- The next step should keep the V4 architecture fixed and tune sampling-time guidance windows and scales before considering new training losses.',
]

summary_path = ARTIFACT_DIR / 'v4a_summary.md'
summary_path.write_text('\n'.join(summary_lines) + '\n')
print(f'Wrote V4a summary to {summary_path}')


In [ ]:
def plot_condition_boxplot(metric: str, title: str, output_path: Path, condition_order: list[str]) -> None:
    """Save a simple boxplot across real, baseline, and guided conditions."""
    frames = [v4a_real_global_diagnostics_df] + [v4a_sample_runs[name]['metrics_df'] for name in condition_order if name != 'real_validation_reference']
    labels = [
        'real',
        'baseline',
        *[
            name.replace('guided_', '').replace('_', '\n')
            for name in condition_order
            if name != 'real_validation_reference' and name != 'baseline_v4'
        ],
    ]
    data = [frame[metric].dropna().to_numpy() for frame in frames]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.boxplot(data, labels=labels, showfliers=False)
    ax.set_title(title)
    ax.set_ylabel(metric)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.show()


def plot_pooled_nonlocal_distance_histograms(output_path: Path, condition_order: list[str]) -> None:
    """Plot pooled nonlocal CA distance distributions for all sampled conditions."""
    fig, ax = plt.subplots(figsize=(10, 5))
    pooled_lookup = {'real_validation_reference': v4a_real_pooled_nonlocal_distances}
    pooled_lookup.update({name: run['pooled_nonlocal_distances'] for name, run in v4a_sample_runs.items()})
    labels = {
        'real_validation_reference': 'real',
        'baseline_v4': 'baseline',
        'guided_thr6_scale1e-4': 'guided thr=6 scale=1e-4',
        'guided_thr8_scale3e-4': 'guided thr=8 scale=3e-4',
        'guided_thr10_scale1e-4': 'guided thr=10 scale=1e-4',
    }
    for condition in ['real_validation_reference', *condition_order]:
        distances = pooled_lookup[condition]
        if len(distances) == 0:
            continue
        bins = np.linspace(0.0, max(30.0, float(np.quantile(distances, 0.995))), 40)
        hist, edges = np.histogram(distances, bins=bins, density=True)
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax.plot(centers, hist, linewidth=2, label=labels.get(condition, condition))
    ax.set_title(f'Nonlocal CA distance distributions (sequence separation > {V4A_NONLOCAL_SEQUENCE_SEPARATION})')
    ax.set_xlabel('CA distance (A)')
    ax.set_ylabel('Density')
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.show()


def plot_close_contact_bar(summary_df: pd.DataFrame, output_path: Path, condition_order: list[str]) -> None:
    """Plot the mean fraction of nonlocal CA pairs below 8 A for each condition."""
    labels = []
    values = []
    subset = summary_df[summary_df['condition'].isin(['real_validation_reference', *condition_order])]
    for condition in ['real_validation_reference', *condition_order]:
        row = subset[subset['condition'] == condition].iloc[0]
        labels.append(
            'real' if condition == 'real_validation_reference' else
            'baseline' if condition == 'baseline_v4' else
            condition.replace('guided_', '').replace('_', '\n')
        )
        values.append(float(row['fraction_nonlocal_ca_pairs_below_8A_mean']))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(labels, values)
    ax.set_title('Mean fraction of nonlocal CA pairs below 8 A')
    ax.set_ylabel('Per-structure mean fraction')
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.show()


def save_representative_samples_figure(
    coords_batch: torch.Tensor,
    mask_batch: torch.Tensor,
    title_prefix: str,
    output_path: Path,
    max_examples: int = 4,
) -> None:
    """Save a 2x2 panel of representative CA traces."""
    n_examples = min(max_examples, coords_batch.shape[0])
    fig = plt.figure(figsize=(10, 10))
    for plot_index in range(n_examples):
        ax = fig.add_subplot(2, 2, plot_index + 1, projection='3d')
        valid_ca = coords_batch[plot_index, mask_batch[plot_index].bool(), 1, :].detach().cpu().numpy()
        ax.plot(valid_ca[:, 0], valid_ca[:, 1], valid_ca[:, 2], marker='o', markersize=2)
        ax.set_title(f'{title_prefix} {plot_index}')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_zlabel('z')
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.show()


plot_condition_order = [config['condition'] for config in V4A_GUIDANCE_EXPERIMENTS]
plot_condition_boxplot(
    'radius_of_gyration',
    'Radius of gyration comparison',
    FIGURE_DIR / 'v4a_radius_of_gyration_comparison.png',
    condition_order=plot_condition_order,
)
plot_pooled_nonlocal_distance_histograms(
    FIGURE_DIR / 'v4a_nonlocal_ca_distance_comparison.png',
    condition_order=plot_condition_order,
)
plot_close_contact_bar(
    v4a_guidance_summary_df,
    FIGURE_DIR / 'v4a_nonlocal_close_contact_fraction_comparison.png',
    condition_order=plot_condition_order,
)
save_representative_samples_figure(
    v4a_sample_runs['baseline_v4']['coords'],
    v4a_sample_masks,
    'Baseline sample',
    FIGURE_DIR / 'v4a_representative_samples_baseline.png',
)
save_representative_samples_figure(
    v4a_sample_runs[V4A_REPRESENTATIVE_GUIDED_CONDITION]['coords'],
    v4a_sample_masks,
    f'Guided sample ({V4A_REPRESENTATIVE_GUIDED_CONDITION})',
    FIGURE_DIR / 'v4a_representative_samples_guided.png',
)

display(v4a_guidance_summary_df)
print(f'V4a summary markdown: {ARTIFACT_DIR / "v4a_summary.md"}')


## 10. V4a outcome summary

V4a is intentionally not a new architecture claim.
It keeps the working V4 denoiser and the stable V3b-style objective fixed, then reframes the remaining failure mode as a global nonlocal-distance problem rather than a purely local-geometry problem.

The intended interpretation is conservative:

- V4 improved local backbone geometry materially, but sampling can still produce globally over-collapsed structures.
- Radius of gyration is useful, but it is not enough on its own because a structure can expand for the wrong reason.
- The main anti-collapse diagnostic in V4a is the nonlocal CA distance distribution, especially the fraction of nonlocal CA pairs below 8 A.
- The sampling-time nonlocal CA guidance term is only a lightweight intervention for controlled comparison; it is not a claim that the training objective has been solved.
- If guided sampling reduces close contacts and collapse counts while preserving adjacent CA spacing and bond geometry, that is evidence for a worthwhile next step.
- If guidance helps radius but hurts local geometry or creates obvious outliers, that trade-off should be reported honestly and treated as a limitation.
